Домашнее задание

ДЗ №5 «Нейросетевые модели»

Цель:

Студенты учатся использовать нейронные сети в моделях торговой стратегии.

Результат - построена модель с использованием нейронных сетей.


Описание/Пошаговая инструкция выполнения домашнего задания:
- включить в модель нейросетевые модели;
- добавить анализ временных рядов;
- построить оценку качетсва модели.

In [ ]:
#!pip install pandas numpy TA-Lib-Precompiled scikit-learn hyperopt plotly backtesting backtrader
#!pip install matplotlib requests_cache
!pip install TA-Lib
#!pip install yfinance_cache
#!pip install fmp_python
#!pip install outlier_utils
!pip install mplfinance
!pip install catboost

In [ ]:

import torch
from scipy.spatial.distance import cosine
from transformers import AutoModel, AutoTokenizer
import gradio as gr

!pip3 install chronos-forecasting
#####!pip3 install git+https://github.com/amazon-science/chronos-forecasting.git
!pip3 install torchmetrics

from chronos import BaseChronosPipeline, Chronos2Pipeline
from torchmetrics.classification import BinaryAUROC

In [ ]:
!pip3 install backtesting
!pip3 install bokeh==3.8.1

In [ ]:
import backtesting
from backtesting import Backtest, Strategy

import bokeh
print(bokeh.__version__)
from backtesting import set_bokeh_output
set_bokeh_output(notebook=True)
backtesting.set_bokeh_output(notebook=True)
if 'google.colab' in str(get_ipython()):
    get_ipython().run_line_magic('matplotlib', 'inline')


3.8.1


/usr/local/lib/python3.12/dist-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import matplotlib.pyplot as plt

import yfinance as yf
import talib as ta
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import joblib

from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew, kurtosis
from statsmodels.tsa.seasonal import STL
import pywt
import mplfinance as mpf
from scipy.fft import fft
from IPython.display import display




In [ ]:
#Машинное обучение
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

Добавляем библиотеки для нейросетевой модели

In [ ]:
import torch.nn as nn
import torch
from torch.utils.data import TensorDataset, DataLoader

from transformers import PatchTSMixerConfig, PatchTSMixerForPrediction

In [ ]:
# backtesting
# -----------------------
class ChronosNextDayDirection(Strategy):
    def init(self):
        # backtesting.py позволяет обращаться к колонкам как self.data.<Name>
        pass

    def next(self):
        sig = int(self.data.Signal[-1])

        # если сигнала нет — выходим в кэш
        if sig == 0:
            if self.position:
                self.position.close()
            return

        # long
        if sig == 1:
            if self.position.is_short:
                self.position.close()
            if not self.position:
                self.buy()

        # short
        if sig == -1:
            if self.position.is_long:
                self.position.close()
            if not self.position:
                self.sell()

In [ ]:
class ChronosStrategy(Strategy):
    def init(self):
        # backtesting.py позволяет обращаться к колонкам как self.data.<Name>
        pass

    def next(self):
        sig = int(self.data.Signal[-1])

        if sig == 0:
            if self.position:
                self.position.close()
            return

        if sig == 1:
            if self.position.is_short:
                self.position.close()
            if not self.position:
                self.buy()

        if sig == -1:
            if self.position.is_long:
                self.position.close()
            if not self.position:
                self.sell()

In [ ]:
class BuyHold(Strategy):
    def init(self):
        # backtesting.py позволяет обращаться к колонкам как self.data.<Name>
        pass

    def next(self):
        if not self.position:
            self.buy()

In [ ]:
def backtestWithChronos(strategy, data, test_data, action=None, instrument=None) :
  print("Обрабатываем данные для модели Chronos... -> ")
  print("=" * 50)

  device_map = "cuda" if torch.cuda.is_available() else "cpu"
  train_data_neu = data.copy()
  test_data_neu = test_data.copy()
  if instrument is not None :
      strategy_instrument = instrument
  if strategy.ticker is not None:
      strategy_instrument = strategy.ticker

  def preprocess_data(df_raw, strategy_instance):
      """Единая функция для обработки train или test данных."""
      df = df_raw.copy()
      # Генерация сигналов
      signals, conditions, datadf = strategy_instance.generate_signals(df)
      if 'Signal' in conditions.columns:
          conditions.drop('Signal', axis=1, inplace=True)
      conditions.columns = conditions.columns.str.capitalize()
      df_signals = signals.reset_index()
      df_signals.columns = ['Date', 'signal']
      datadf = pd.merge(datadf, df_signals, on='Date', how='inner').fillna(0)
      # Удаляем старую колонку 'Signal', если она есть
      if 'Signal' in datadf.columns:
          datadf.drop('Signal', axis=1, inplace=True)
      datadf.rename(columns={'Date': 'Datetime'}, inplace=True)
      datadf["Datetime"] = pd.to_datetime(datadf["Datetime"])
      datadf.set_index('Datetime', inplace=True)
      # Приводим все колонки к одному регистру (например, Capitalize)
      datadf.columns = datadf.columns.str.capitalize()
      # Расчет доходностей (используем ffill для заполнения первого NaN)
      datadf['Returns'] = datadf['Close'].pct_change()
      datadf['Returns'] = datadf['Returns'].ffill().fillna(0.0)
      # Заполнение оставшихся NaN/Inf нулями для безопасности
      datadf = datadf.replace([np.inf, -np.inf], np.nan).fillna(0)
      # Подготовка таргета
      datadf['y_next_close'] = datadf['Close'].shift(-1)
      # Удаляем последнюю строку, где нет таргета
      ## datadf = datadf.iloc[:-1].copy()
      # Создаем бинарный таргет
      datadf['target'] = (datadf['y_next_close'] > datadf['Close']).astype(int)
      return datadf, conditions # Возвращаем DataFrame и список доп. признаков

  datadf_neu, train_conditions_neu = preprocess_data(train_data_neu, strategy)
  ##
  datadf_neu = datadf_neu.iloc[:-1].copy()
  datadf_neu = datadf_neu.replace([np.inf, -np.inf], np.nan)
  datadf_neu.columns = datadf_neu.columns.str.capitalize()

  FEATURES = ['Close', 'Volume', 'Returns']
  train_conditions_neu = train_conditions_neu.reindex(columns=FEATURES + train_conditions_neu.columns.tolist())
  FEATURES = train_conditions_neu.columns

  # Rolling-прогноз: завтра (t+1)
  # -----------------------
  print("Выставляем параметры для модели Chronos... -> ")
  LOOKBACK = 128            # сколько последних дней даём модели как "контекст"
  THRESH   = 0.002          # порог для сигнала (0.2%) чтобы отсечь шум
  PRED_LEN = 1

  tail_size = LOOKBACK + 1
  test_data_extended = pd.concat([train_data_neu.iloc[-tail_size:], test_data_neu])
  test_datadf_neu, test_conditions_neu = preprocess_data(test_data_extended, strategy)
  close = datadf_neu["Close"].values.astype(np.float32)

  print("Pretrain BaseChronosPipeline... ->")
  print("=" * 50)

  chronos_output_dir = "./my_chronos_model"+strategy_instrument
  # Сохранение весов и конфигурации
  #model.save_pretrained(output_dir)
  # Сохранение токенизатора (необходимо для инференса)
  #tokenizer.save_pretrained(output_dir)



  # Загрузка локальной модели
  #pipeline = Chronos2Pipeline.from_pretrained("./my_chronos_model", device_map="cuda")

  chronos_output_dir = "./my_chronos_model"+strategy_instrument
  pipeline = BaseChronosPipeline.from_pretrained(
      "amazon/chronos-bolt-tiny",
      device_map=device_map,
      dtype=torch.bfloat16 if device_map != "cpu" else torch.float32,
  )
  if (action == "Load") and os.path.isdir(chronos_output_dir):
      pipeline = BaseChronosPipeline.from_pretrained(chronos_output_dir, device_map=device_map)

  # найдём индекс квантиля 0.5 (медиана), если доступно
  def get_median_forecast(forecast_tensor, pipeline_obj):
      # forecast_tensor shape: [1, Q, 1]
      Q = forecast_tensor.shape[1]
      q_list = getattr(pipeline_obj, "quantiles", None)
      if q_list is not None and 0.5 in q_list:
          qi = q_list.index(0.5)
      else:
          # fallback: берём центральный квантиль
          qi = Q // 2
      return float(forecast_tensor[0, qi, 0].detach().cpu().float().numpy())


  signals = np.zeros(len(datadf_neu), dtype=np.int8)   # -1 short, 0 flat, +1 long
  pred_next = np.full(len(datadf_neu), np.nan, dtype=np.float32)

  # сигнал на день t рассчитываем по инфо до t включительно, торгуем на t+1
  #for t in range(LOOKBACK, len(datadf_neu) - 1):
  #    ctx = torch.tensor(close[t - LOOKBACK : t + 1])  # 1D context
  #    with torch.no_grad():
  #        fc = pipeline.predict(ctx, prediction_length=PRED_LEN)  # [1, Q, 1]
  #    yhat = get_median_forecast(fc, pipeline)
  #    pred_next[t] = yhat
  #
  #    today = close[t]
  #    if yhat > today * (1.0 + THRESH):
  #        signals[t] = 1
  #    elif yhat < today * (1.0 - THRESH):
  #        signals[t] = -1
  #    else:
  #        signals[t] = 0
  for t in range(LOOKBACK, len(datadf_neu) - 1):
      ##ctx = torch.tensor(close[t - LOOKBACK : t + 1])  # 1D context
      ctx = torch.tensor(close[t - LOOKBACK : t + 1]).unsqueeze(0).to(device_map)
      if device_map == "cuda":
          ctx = ctx.to(torch.bfloat16)
      with torch.no_grad():
          fc = pipeline.predict(ctx, prediction_length=PRED_LEN)  # [1, Q, 1]
      yhat = get_median_forecast(fc, pipeline)
      pred_next[t] = yhat

      today = close[t]
      if yhat > today * (1.0 + THRESH):
          signals[t] = 1
      elif yhat < today * (1.0 - THRESH):
          signals[t] = -1
      else:
          signals[t] = 0

  datadf_neu["Signal"] = signals
  datadf_neu["PredNextClose"] = pred_next

  pipeline.model.save_pretrained(chronos_output_dir)

  print("=" * 50)
  print("ChronosNextDayDirection... -> ")
  bt_nextday = Backtest(
      datadf_neu, ChronosNextDayDirection,
      cash=1000_000,
      commission=0.002,   # 0.2% комиссия
      trade_on_close=True, # сделка по close текущего бара
      finalize_trades=True
  )
  #!!!!!!!!2.3 Look-ahead в исполнении сигнала
  #Сейчас Signal применяется на той же свече, по которой считается Close, а прибыль считается от Close[t] -> Close[t+1].
  #Если сигнал формируется с использованием Close[t] (и индикаторов на нём), то торговля “по close” в конце свечи
  # - это оптимистичное допущение (в реальности ты узнаёшь close в момент закрытия)
  #, хотя "в принципе" мы можем достаточно хорошо ориентироваться на "Close" закрывая позицию в конце свечи.
  #Тут просто как рекомендация для более "честной" стратегии.
  # =====>>>> В библиотеках для бэктестинга (например, backtrader или vectorbt) это обычно настраивается параметрами:
  #Вместо trade_on_close=True (как у вас сейчас), используйте настройку для торговли на открытии следующего бара (trade_on_open=True или аналогичную).
  #Убедитесь, что ваша стратегия использует данные бара [t] для расчета сигнала, который затем применяется к бару [t+1].
  #Это значительно повысит реалистичность вашего тестирования, хоть и может немного снизить итоговый Sharpe Ratio.

  stats_nextday = bt_nextday.run()

 # def chronos_predict_next_close(df_chronos, pipeline, lookback=LOOKBACK):
 #   close = df_chronos["Close"].values.astype(np.float32)
 #   preds = np.full(len(df_chronos), np.nan)
 #
 #   for t in range(lookback, len(df_chronos) - 1):
 #       ctx = torch.tensor(
 #           close[t - lookback : t + 1],
 #       )
 #       with torch.no_grad():
 #           fc = pipeline.predict(ctx, prediction_length=1)
 #
 #       preds[t] = get_median_forecast(fc, pipeline)
 #
 #   return preds

  def chronos_predict_next_close(df_chronos, pipeline, lookback=LOOKBACK):
    close = df_chronos["Close"].values.astype(np.float32)
    preds = np.full(len(df_chronos), np.nan, dtype=np.float32)

    pipeline.model.eval()
    device = next(pipeline.model.parameters()).device
    #dtype = torch.bfloat16 if device_map != "cpu" else torch.float32

    #for t in range(lookback, len(df_chronos) - 1):
    for t in range(lookback, len(df_chronos)):
        ctx_slice = close[t - lookback : t + 1]
        ctx = torch.tensor(ctx_slice, device=device).unsqueeze(0)
        if device.type == "cuda":
            ctx = ctx.to(torch.bfloat16)
        with torch.no_grad():
            fc = pipeline.predict(ctx, prediction_length=1)
        preds[t] = get_median_forecast(fc, pipeline)

        #ctx = torch.tensor(
        #    close[t - lookback : t + 1],
        #).unsqueeze(0).to(device_map)
        #with torch.no_grad():
        #    fc = pipeline.predict(ctx, prediction_length=1)

        #preds[t] = get_median_forecast(fc, pipeline)

    return preds

  datadf_neu["PredNext"] = chronos_predict_next_close(
    datadf_neu, pipeline, LOOKBACK
  )

  datadf_neu = datadf_neu.dropna(subset=["PredNext"])

  #================================================
  # Расчет ROC AUC для модели
  def calculate_chronos_roc_auc(df):
    # Очищаем от NaN (где не хватило LOOKBACK)
    valid_df = df.dropna(subset=["PredNext"]).copy()

    # 1. Реальное движение цены (Target): 1 если выросла, 0 если упала или осталась той же
    # Используем Close завтрашнего дня относительно Close сегодня
    actual_move = (valid_df["Close"].shift(-1) > valid_df["Close"]).astype(int)

    # 2. Прогноз модели (Score): Ожидаемое изменение цены
    # (Чем больше разница, тем выше уверенность модели в росте)
    predicted_change = valid_df["PredNext"] - valid_df["Close"]

    # Удаляем последнюю строку, так как для неё нет "завтрашнего" Close
    actual_move = actual_move[:-1]
    predicted_change = predicted_change[:-1]

    if len(actual_move) > 0:
        auc = roc_auc_score(actual_move, predicted_change)
        return auc
    return None

  # Применение для обучающей выборки
  print("=" * 70)
  train_auc = calculate_chronos_roc_auc(datadf_neu)
  print(f"ROC AUC (Train): {train_auc:.4f}")
  print("=" * 70)

  #==============================
  #!!!!!!!!!1. В ячейке 13 есть блок:
  #Next_Ret = Close.pct_change().shift(-1)
  #и перебор candidates и выбор best_thr по Sharpe
  #Если порог выбирается на том же периоде, на котором потом оцениваешь стратегию - это overfitting на метрику.
  #Порог лучше подбирать на отдельном “validation” внутри train (например, 80/20 по времени внутри train), а финально мерить только на test/valid.
  # =====>>>>>
  # --- 1. Разделение TRAIN на Train-Proper и Validation ---
  # Берем datadf_neu (который получен из train_data_neu)
  #split_idx = int(len(datadf_neu) * 0.8)
  #train_subset = datadf_neu.iloc[:split_idx].copy()
  #val_subset = datadf_neu.iloc[split_idx:].copy()
  #print(f"Подбор порога на валидации: {len(val_subset)} свечей")
  # --- 2. Получаем прогнозы для валидационной части ---
  # Чтобы не пересчитывать всё, можно взять уже готовые PredNext из datadf_neu
  # если они были посчитаны для всего набора ранее.
  #val_subset["Next_Ret"] = val_subset["Close"].pct_change().shift(-1)
  #val_subset = val_subset.dropna(subset=["PredNext", "Next_Ret"])
  #candidates = np.linspace(0.0, 0.01, 21) # 0% - 1%
  #best_thr = 0.0
  #best_sharpe = -np.inf
  #for thr in candidates:
  #    # Генерируем сигнал только на валидационном наборе
  #    cond_long = val_subset["PredNext"] > val_subset["Close"] * (1 + thr)
  #    cond_short = val_subset["PredNext"] < val_subset["Close"] * (1 - thr)
  #   val_signal = np.where(cond_long, 1, np.where(cond_short, -1, 0))
  #   # Доходность стратегии на валидации
  #   strat_ret = val_signal * val_subset["Next_Ret"]
  #   # Считаем коэффициент Шарпа
  #   if strat_ret.std() == 0:
  #        sharpe = 0
  #   else:
  #        sharpe = (strat_ret.mean() / strat_ret.std()) * np.sqrt(252) # Годовое значение
  #    if sharpe > best_sharpe:
  #        best_sharpe = sharpe
  #        best_thr = thr
  #print(f"Выбран порог {best_thr:.4f} (Sharpe на валидации: {best_sharpe:.4f})")
  # --- 3. Финальное применение на TEST ---
  # Теперь используем best_thr, который модель "не видела" в контексте доходности теста
  #test_datadf_neu["Signal"] = np.where(
  #    test_datadf_neu["PredNext"] > test_datadf_neu["Close"] * (1 + best_thr), 1,
  #    np.where(test_datadf_neu["PredNext"] < test_datadf_neu["Close"] * (1 - best_thr), -1, 0)
  #)
  #returns = datadf_neu["Close"].pct_change().shift(-1)
  #datadf_neu["Next_Ret"] = datadf_neu["Close"].pct_change().shift(-1)
  #datadf_neu_clean = datadf_neu.dropna(subset=["PredNext", "Next_Ret"])
  #candidates = np.linspace(0.0, 0.01, 21)  # 0% … 1%
  ###best_thr, best_sharpe = None, -np.inf

  #for thr in candidates:
  #    signal = np.where(
  #        datadf_neu["PredNext"] > datadf_neu["Close"] * (1 + thr),  1,
  #        np.where(datadf_neu["PredNext"] < datadf_neu["Close"] * (1 - thr), -1, 0)
  #    )
  #
  #    strat_ret = signal * returns
  #    sharpe = strat_ret.mean() / strat_ret.std()
  #
  #    if sharpe > best_sharpe:
  #        best_sharpe = sharpe
  #        best_thr = thr

  for thr in candidates:
      cond_long = datadf_neu_clean["PredNext"] > datadf_neu_clean["Close"] * (1 + thr)
      cond_short = datadf_neu_clean["PredNext"] < datadf_neu_clean["Close"] * (1 - thr)
      temp_signal = np.where(cond_long, 1, np.where(cond_short, -1, 0))
      strat_ret = temp_signal * datadf_neu_clean["Next_Ret"]
      # Расчет шарпа (упрощенно)
      sharpe = strat_ret.mean() / (strat_ret.std() + 1e-9)
      if sharpe > best_sharpe:
          best_sharpe = sharpe
          best_thr = thr

  print("Best threshold:", best_thr, "Sharpe:", best_sharpe)

  test_datadf_neu["PredNext"] = chronos_predict_next_close(
    test_datadf_neu, pipeline, LOOKBACK
  )

  test_datadf_neu["Signal"] = np.where(
      test_datadf_neu["PredNext"] > test_datadf_neu["Close"] * (1 + best_thr),  1,
      np.where(test_datadf_neu["PredNext"] < test_datadf_neu["Close"] * (1 - best_thr), -1, 0)
  )

  print("=" * 50)
  print("ChronosStrategy... -> ")
  bt_chronos = Backtest(
      test_datadf_neu,
      ChronosStrategy,
      cash=1000_000,
      commission=0.002,
      trade_on_close=True,
      finalize_trades=True
  )

  stats_chronos = bt_chronos.run()

  print("=" * 50)
  print("BuyHold... -> ")
  bt_bh = Backtest(
    test_datadf_neu,
    BuyHold,
    cash=1000_000,
    commission=0.002,
    trade_on_close=True,
    finalize_trades=True
  )

  stats_bh = bt_bh.run()

  print("=" * 50)
  print("TemaGoldenCrossStrategy... -> ")
  #test_datadf_neu = test_datadf_neu[ LOOKBACK: ]
  #test_datadf_neu_signals = test_datadf_neu[['Signal']].copy()
  #test_datadf_neu_signals.index.name = 'Date'
  #test_datadf_neu_signals.columns = ['signal']
  #test_datadf_neu['signal'] = test_datadf_neu_signals['signal'].copy()

  test_datadf_neu = test_datadf_neu[ LOOKBACK: ]
  test_datadf_neu_signals = test_datadf_neu[['Signal']].copy()
  test_datadf_neu_signals.index.name = 'Date'
  test_datadf_neu_signals.columns = ['signal']
  test_datadf_neu['Signal'] = test_datadf_neu_signals['signal']
  test_datadf_neu['Signal'] = test_datadf_neu['Signal'].fillna(0)
  test_datadf_neu_signals = test_datadf_neu['Signal']

  test_datadf_neu_backtest = setBktDataFrame(test_datadf_neu_signals, test_conditions_neu, test_datadf_neu)


  #test_datadf_neu_backtest['Signal'] = test_datadf_neu_backtest['Label']
  btchronos = Backtest(test_datadf_neu, TemaGoldenCrossStrategy, cash=1000000, commission=.002, exclusive_orders=True, finalize_trades=True)
  stats_goldencross = btchronos.run()
  display("GoldenCrossStrategy:  ")
  btchronos.plot(
        plot_equity=True,
        plot_drawdown=True,
        relative_equity=False,
        superimpose=False #при date = 1h ошибка Invalid value for `superimpose`: Upsampling not supported
    )
  #bt.plot()
  #print(stats_goldencross[:27])

  db_chronos = pd.concat([stats_bh, stats_nextday], axis = 1)
  db_chronos.columns = ['Buy&Hold', 'GoldenCross']
  display(db_chronos)
  comparison = pd.DataFrame({
    "Chronos": stats_nextday,
    #"GoldenCross": stats_goldencross,
    "Buy&Hold": stats_bh
  })

  #print(comparison)

  risk_adjusted_return_chronos = (
            stats_nextday.iloc[7] *
            stats_nextday.iloc[12] *
            stats_nextday.iloc[13] *
            stats_nextday.iloc[7]
        ) / (1 + stats_nextday.iloc[17])

  display(f'risk_adjusted_return = {risk_adjusted_return_chronos:.2f}')

  #y_pred = torch.tensor(y_pred_binary)
  #y_true = torch.tensor(y_true_binary)
  #metric = BinaryAUROC()
  #auc_score = metric(y_pred, y_true)
  #print(f"ROC AUC: {auc_score.item()}")

  return_chronos = {
            'final_equity': stats_nextday.iloc[4], #final_equity,
            'n_trades': stats_nextday.iloc[21], #n_trades,
            'returns': stats_nextday.iloc[7], #returns,
            'max_drawdown': abs(stats_nextday.iloc[17]), #max_drawdown,
            'win_rate': stats_nextday.iloc[22], #win_rate,
            'sharpe_ratio': stats_nextday.iloc[12], #sharpe_ratio,
            'sortino_ratio': stats_nextday.iloc[13], #sortino_ratio,
            'profit_factor': stats_nextday.iloc[7], #profit_factor
            'pipeline': pipeline,
            'strategy_ticker': strategy.ticker
            #'rocauc': roc, #rocauc для модели
            #'pred': y_pred_binary.copy(), # запоминаем предсказание для набора
        }

  return return_chronos

In [ ]:
# Определение модели классификации

class TSMixerBinaryClassifier(nn.Module):
    """
    Бинарный классификатор на основе PatchTSMixer.

    Архитектура:
    1. Backbone: стандартный PatchTSMixerForPrediction из HuggingFace
    2. Feature extraction: используем prediction_outputs как признаки
    3. Classification head: полносвязные слои с dropout

    Преимущества:
    - Используем проверенную архитектуру из transformers
    - Backbone уже оптимизирован для работы с временными рядами
    - Простота и прозрачность кода
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Backbone: стандартная модель PatchTSMixer для prediction
        # Она обучается предсказывать следующий шаг, и мы используем её выходы как признаки
        self.backbone = PatchTSMixerForPrediction(config)

        # Classification head
        # Входные признаки: (batch, prediction_length=1, num_channels)
        # После flatten: (batch, num_channels)
        self.classifier = nn.Sequential(
            nn.Flatten(),  # (batch, 1, num_channels) -> (batch, num_channels)
            nn.Linear(config.num_input_channels, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)  # 2 класса: падение (0) и рост (1)
        )

    def forward(self, past_values):
        """
        Forward pass.

        Параметры:
        - past_values: (batch, context_length, num_channels)

        Возвращает:
        - logits: (batch, 2) - логиты для двух классов
        """
        # Получаем prediction outputs из backbone
        outputs = self.backbone(past_values=past_values)

        # prediction_outputs имеет размер (batch, prediction_length=1, num_channels)
        # Эти выходы содержат закодированную информацию о временном ряде
        features = outputs.prediction_outputs

        # Пропускаем через classification head
        logits = self.classifier(features)

        return logits

print("Класс TSMixerBinaryClassifier определен")

Класс TSMixerBinaryClassifier определен


In [ ]:
def setBktDataFrame(df_signals, df_conditions, df_data_all) :

  df_backtest = df_data_all.copy()

  # Убеждаемся, что индекс назван 'Date' или 'Datetime' и является временным
  if df_backtest.index.name is None:
     df_backtest.index.name = 'Datetime'

  # Все колонки должны быть Capitalize для ChronosNextDayDirection
  df_backtest.columns = df_backtest.columns.str.capitalize()

  # Библиотека Backtest ожидает колонку 'Open', 'High', 'Low', 'Close', 'Volume' и 'Signal'
  required_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Signal']

  # Проверка наличия всех нужных колонок
  if not all(col in df_backtest.columns for col in required_cols):
     missing = [col for col in required_cols if col not in df_backtest.columns]
     raise ValueError(f"Отсутствуют обязательные колонки для Backtest: {missing}")

  return df_backtest[required_cols] # Возвращаем только нужные колонки

In [ ]:
def backtestWithNeu(strategy, data, test_data, action=None, instrument=None) :

  train_data_neu = data.copy()
  test_data_neu = test_data.copy()
  if instrument is not None :
      strategy_instrument = instrument
      tsmixer_output_dir = "tsmixer_full_"+strategy_instrument+".pt"
  if strategy.ticker is not None:
      strategy_instrument = strategy.ticker
      tsmixer_output_dir = "tsmixer_full_"+strategy_instrument+".pt"

  def preprocess_data(df_raw, strategy_instance):
    """Единая функция для обработки train или test данных."""
    df = df_raw.copy()
    # Генерация сигналов
    signals, conditions, datadf = strategy_instance.generate_signals(df)
    if 'Signal' in conditions.columns:
        conditions.drop('Signal', axis=1, inplace=True)
    conditions.columns = conditions.columns.str.capitalize()
    df_signals = signals.reset_index()
    df_signals.columns = ['Date', 'signal']
    datadf = pd.merge(datadf, df_signals, on='Date', how='inner').fillna(0)
    # Удаляем старую колонку 'Signal', если она есть
    if 'Signal' in datadf.columns:
        datadf.drop('Signal', axis=1, inplace=True)
    datadf.rename(columns={'Date': 'Datetime'}, inplace=True)
    datadf["Datetime"] = pd.to_datetime(datadf["Datetime"])
    datadf.set_index('Datetime', inplace=True)
    # Приводим все колонки к одному регистру (например, Capitalize)
    datadf.columns = datadf.columns.str.capitalize()
    # Расчет доходностей (используем ffill для заполнения первого NaN)
    datadf['Returns'] = datadf['Close'].pct_change()
    datadf['Returns'] = datadf['Returns'].ffill().fillna(0.0)
    # Заполнение оставшихся NaN/Inf нулями для безопасности
    datadf = datadf.replace([np.inf, -np.inf], np.nan).fillna(0)
    # Подготовка таргета
    datadf['y_next_close'] = datadf['Close'].shift(-1)
    # Удаляем последнюю строку, где нет таргета
    datadf = datadf.iloc[:-1].copy()
    # Создаем бинарный таргет
    datadf['target'] = (datadf['y_next_close'] > datadf['Close']).astype(int)
    return datadf, conditions # Возвращаем DataFrame и список доп. признаков

  # Шаг 1: Подготовка данных для бинарной классификации
  # ======================================================

  LOOKBACK = 128        # сколько дней истории подаем в модель
  PATCHLENGTH = LOOKBACK - 1
  HORIZON  = 7         # сколько дней прогнозируем вперед
  TRAIN_RATIO = 1
  EPOCHS = 50
  BATCH_SIZE = 64
  LR = 3e-4
  WEIGHT_DECAY = 1e-2
  SEED = 42
  device = "cuda" if torch.cuda.is_available() else "cpu"

  #=======================================================
  df_binary, train_conditions_neu = preprocess_data(train_data_neu, strategy)
  FEATURES = ['Returns', 'Close', 'Volume' ]
  train_conditions_neu = train_conditions_neu.reindex(columns=FEATURES + train_conditions_neu.columns.tolist())
  FEATURES = train_conditions_neu.columns

  tail_size = LOOKBACK + 1
  test_data_extended = pd.concat([train_data_neu.iloc[-tail_size:], test_data_neu])
  test_datadf_neu, test_conditions_neu = preprocess_data(test_data_extended, strategy)
  ## --- ШАГ 1: Поиск параметров на TRAIN (через внутреннюю валидацию) ---
  # Находим параметры, используя только тренировочные данные
  #best_params = optimize_on_validation(train_data_neu)
  # --- ШАГ 2: Подготовка ТЕСТА (с контекстом из прошлого) ---
  # Для Chronos нам нужны последние LOOKBACK свечей из Train, чтобы предсказать начало Test
  #tail_size = best_params['LOOKBACK']
  #test_data_extended = pd.concat([train_data_neu.iloc[-tail_size:], test_data_neu])
  # --- ШАГ 3: Инференс на ТЕСТЕ ---
  # Используем параметры, найденные на шаге 1. НИКАКОЙ новой оптимизации!
  #test_preds = get_chronos_predictions(test_data_extended, pipeline, best_params)
  # --- ШАГ 4: Финальный Бэктест ---
  #stats_test = run_backtest(test_data_neu, test_preds, best_params)

  #=======================================================
  print("=" * 70)
  print("РАСПРЕДЕЛЕНИЕ КЛАССОВ")
  print("=" * 70)
  print(df_binary['target'].value_counts().sort_index())
  print(f"\nБаланс классов (доля класса 1): {df_binary['target'].mean():.3f}")
  print(f"Всего примеров: {len(df_binary)}")
  print("=" * 70)

  # 2: Нормализация и создание временных окон
  # ================================================

  # Извлекаем признаки и таргет
  X_binary = df_binary[FEATURES].astype("float32").to_numpy()
  y_binary = df_binary["target"].astype("int64").to_numpy()

  X_binary_test = test_datadf_neu[FEATURES].astype("float32").to_numpy()
  y_binary_test = test_datadf_neu["target"].astype("int64").to_numpy()
  scaler = MinMaxScaler(feature_range=(0, 1))

  #X_binary = scaler.fit_transform(X_binary)
  #X_binary_test = scaler.transform(X_binary_test)

  # Нормализация признаков (только по train!)
  mu_binary = X_binary.mean(axis=0, keepdims=True)
  sd_binary = X_binary.std(axis=0, keepdims=True) + 1e-8
  X_binary_norm = (X_binary - mu_binary) / sd_binary

  # Нормализация признаков (только по test!)
  X_binary_norm_test = (X_binary_test - mu_binary) / sd_binary

  # Функция для создания скользящих окон
  def create_classification_windows(X_normalized, y_labels, lookback):
      """
      Создает скользящие окна для задачи классификации временных рядов.

      Параметры:
      - X_normalized: нормализованные признаки (T, num_features)
      - y_labels: метки классов (T,)
      - lookback: размер окна истории

      Возвращает:
      - X_windows: (N, lookback, num_features)
      - y_windows: (N,)
      - indices: индексы в оригинальном массиве
      """
      X_windows, y_windows, indices = [], [], []

      for i in range(lookback, len(y_labels)):
          # Берем окно истории размером lookback
          X_windows.append(X_normalized[i - lookback:i, :])
          # Таргет - это метка в момент времени i
          y_windows.append(y_labels[i])
          indices.append(i)

      X_windows = np.asarray(X_windows, dtype=np.float32)  # (N, lookback, num_features)
      y_windows = np.asarray(y_windows, dtype=np.int64)     # (N,)
      indices = np.asarray(indices, dtype=np.int64)

      return X_windows, y_windows, indices

  # Создаем окна
  X_win, y_win, idx_win = create_classification_windows(X_binary_norm, y_binary, LOOKBACK)
  # Создаем окна test
  X_win_test, y_win_test, idx_win_test = create_classification_windows(X_binary_norm_test, y_binary_test, LOOKBACK)


  print(f"Размер окон признаков: {X_win.shape}")
  print(f"Размер таргетов: {y_win.shape}")
  print(f"Размер индексов: {idx_win.shape}")

  # 3: Разделение на train/test
  # ==================================

  # Разделяем данные
  X_train_binary = X_win
  y_train_binary = y_win
  idx_train_binary = idx_win

  # Разделяем данные test
  X_test_binary = X_win_test
  y_test_binary = y_win_test
  idx_test_binary = idx_win_test

  print("=" * 70)
  print("РАЗМЕРЫ ВЫБОРОК")
  print("=" * 70)
  print(f"Train: X={X_train_binary.shape}, y={y_train_binary.shape}")
  print(f"Test:  X={X_test_binary.shape}, y={y_test_binary.shape}")
  print(f"\nРаспределение классов в test:")
  print(f"  Класс 0 (падение): {(y_test_binary == 0).sum()} ({(y_test_binary == 0).mean():.1%})")
  print(f"  Класс 1 (рост):    {(y_test_binary == 1).sum()} ({(y_test_binary == 1).mean():.1%})")
  print("=" * 70)

  # 4: Создание DataLoaders
  # =============================

  train_dataset_binary = TensorDataset(
      torch.from_numpy(X_train_binary),
      torch.from_numpy(y_train_binary)
  )

  test_dataset_binary = TensorDataset(
      torch.from_numpy(X_test_binary),
      torch.from_numpy(y_test_binary)
  )

  train_loader_binary = DataLoader(
      train_dataset_binary,
      batch_size=BATCH_SIZE,
      shuffle=True,
      drop_last=False
  )

  test_loader_binary = DataLoader(
      test_dataset_binary,
      batch_size=BATCH_SIZE,
      shuffle=False,
      drop_last=False
  )

  print(f"Train batches: {len(train_loader_binary)}")
  print(f"Test batches:  {len(test_loader_binary)}")

  # Проверим размерность одного батча
  xb_sample, yb_sample = next(iter(train_loader_binary))
  print(f"\nПример батча:")
  print(f"  X: {xb_sample.shape}  (batch, lookback, num_features)")
  print(f"  y: {yb_sample.shape}  (batch,)")

  # Проверим размерность одного батча test
  xb_sample_test, yb_sample_test = next(iter(test_loader_binary))
  print(f"\nПример батча test:")
  print(f"  X: {xb_sample_test.shape}  (batch, lookback, num_features)")
  print(f"  y: {yb_sample_test.shape}  (batch,)")

  # 6: Создание и инициализация модели
  # ========================================

  # Конфигурация модели
  # ВАЖНО: prediction_length=1, так как мы используем выход как feature vector
  config_binary = PatchTSMixerConfig(
      context_length=LOOKBACK,          # Длина входного окна
      prediction_length=1,              # Выход размером 1 (используется как признаки)
      #Обычно для классификации временных рядов prediction_length устанавливается в 0 или игнорируется, если используется PatchTSMixerForClassification
      num_input_channels=len(FEATURES), # Количество признаков (12)
      patch_length=PATCHLENGTH,                  # Размер патча
      #patch_stride=8,                   # Шаг между патчами
      patch_stride=PATCHLENGTH,
      #Обычно stride выбирают равным patch_length (без перекрытия) или чуть меньше. С таким маленьким шагом модель будет обучаться очень медленно.
      d_model=96,                       # Размерность скрытого представления
      num_layers=6,                     # Количество слоев
      dropout=0.1,                      # Dropout для регуляризации
  )


  # Создаем модель
  print("=" * 70)
  model_binary = TSMixerBinaryClassifier(config_binary).to(device)
  if (action=="Load") & os.path.exists(tsmixer_output_dir):
     print(f"Load {tsmixer_output_dir} ...")
     #model_binary = torch.load(tsmixer_output_dir, weights_only=False)
     #model_binary.load_state_dict(torch.load(tsmixer_output_dir, weights_only=True))
     checkpoint = torch.load(tsmixer_output_dir, map_location=device)
     model_binary.load_state_dict(checkpoint['model_state_dict'])
     optimizer_binary = torch.optim.AdamW(model_binary.parameters(), lr=LR)
     if 'optimizer_state_dict' in checkpoint:
        optimizer_binary.load_state_dict(checkpoint['optimizer_state_dict'])

     model_binary.to(device)
     model_binary.eval() # Перевод в режим предсказания
     EPOCHS=0
     print(f"Модель {tsmixer_output_dir} загружена.")

  if os.path.exists(tsmixer_output_dir):
      print(f"Загружены лучшие веса из {tsmixer_output_dir}")

  # Считаем количество параметров
  total_params = sum(p.numel() for p in model_binary.parameters())
  trainable_params = sum(p.numel() for p in model_binary.parameters() if p.requires_grad)

  print("=" * 70)
  print("ИНФОРМАЦИЯ О МОДЕЛИ")
  print("=" * 70)
  print(f"Всего параметров:      {total_params:,}")
  print(f"Обучаемых параметров:  {trainable_params:,}")
  print(f"Устройство:            {device}")
  print("=" * 70)

  # 7: Настройка обучения
  # ===========================

  # Optimizer: AdamW с weight decay для регуляризации
  if (action!="Load"):
      print(f"Установлен новый Optimizer: AdamW с weight decay для регуляризации")
      optimizer_binary = torch.optim.AdamW(
          model_binary.parameters(),
          lr=LR,
          weight_decay=WEIGHT_DECAY
      )

  # -------------------------------
  # 1. Считаем количество примеров каждого класса в обучающей выборке
  class_counts = df_binary['target'].value_counts().sort_index().values
  # class_counts[0] — количество объектов класса 0 (Down)
  # class_counts[1] — количество объектов класса 1 (Up)

  # 2. Вычисляем веса (обратная пропорция)
  # Общая логика: weight = total_samples / (n_classes * class_samples)
  total_samples = len(df_binary)
  n_classes = 2

  weights = total_samples / (n_classes * class_counts)

  # 3. Переводим в тензор и отправляем на устройство (CPU/GPU)
  weights_tensor = torch.tensor(weights, dtype=torch.float32).to(device)

  # 4. Инициализируем лосс с весами
  criterion_binary = nn.CrossEntropyLoss(weight=weights_tensor)

  print(f"Веса классов для Loss: 0 (Down): {weights[0]:.4f}, 1 (Up): {weights[1]:.4f}")


  # Loss function: CrossEntropyLoss для бинарной классификации
  ##criterion_binary = nn.CrossEntropyLoss()
  #Вы используете стандартный CrossEntropyLoss(). Если на шаге подготовки данных вы увидели дисбаланс (например, класс 1 занимает 60%), модель может просто научиться всегда предсказывать единицу.
  #criterion_binary = nn.CrossEntropyLoss(weight=torch.tensor([weight0, weight1]).to(device))


  # Функция для evaluation
  def evaluate_binary_classifier():
      """
      Оценка модели на test set.

      Возвращает:
      - accuracy: точность классификации
      - predictions: предсказанные метки
      - true_labels: истинные метки
      """
      model_binary.eval()
      all_preds = []
      all_labels = []

      with torch.no_grad():
          for xb, yb in test_loader_binary:
              xb = xb.to(device)
              yb = yb.to(device)

              # Forward pass
              logits = model_binary(xb)

              # Предсказания (argmax по логитам)
              preds = torch.argmax(logits, dim=1)

              # Сохраняем для подсчета метрик
              all_preds.extend(preds.cpu().numpy())
              all_labels.extend(yb.cpu().numpy())

      all_preds = np.array(all_preds)
      all_labels = np.array(all_labels)

      accuracy = (all_preds == all_labels).mean()

      return accuracy, all_preds, all_labels

  print("✓ Optimizer, loss function и evaluation function настроены")

  # 8: Обучение модели
  # ========================
  # Инициализация для сохранения лучшей модели
  best_test_acc = 0.0

  print("=" * 70)
  print("НАЧАЛО ОБУЧЕНИЯ")
  print("=" * 70)

  for epoch in range(1, EPOCHS + 1):
      # Training phase
      model_binary.train()
      train_loss_sum = 0.0
      train_correct = 0
      train_total = 0

      for xb, yb in train_loader_binary:
          xb = xb.to(device)
          yb = yb.to(device)

          # Forward pass
          logits = model_binary(xb)
          loss = criterion_binary(logits, yb)

          # Backward pass
          optimizer_binary.zero_grad(set_to_none=True)
          loss.backward()

          # Gradient clipping для стабильности
          torch.nn.utils.clip_grad_norm_(model_binary.parameters(), max_norm=1.0)

          # Update weights
          optimizer_binary.step()

          # Статистика
          train_loss_sum += loss.item()
          preds = torch.argmax(logits, dim=1)
          train_correct += (preds == yb).sum().item()
          train_total += yb.size(0)

      # Метрики за эпоху
      train_loss_avg = train_loss_sum / len(train_loader_binary)
      train_acc = train_correct / train_total

      # Evaluation на test set
      test_acc, _, _ = evaluate_binary_classifier()

      # ЛОГИКА СОХРАНЕНИЯ ЛУЧШЕЙ МОДЕЛИ
      if test_acc > best_test_acc:
          best_test_acc = test_acc
          # Сохраняем состояние (state_dict) - самый надежный способ
          #torch.save(model_binary.state_dict(), tsmixer_output_dir)
          checkpoint = {
              'model_state_dict': model_binary.state_dict(),
              'optimizer_state_dict': optimizer_binary.state_dict(),
          }
          torch.save(checkpoint, tsmixer_output_dir)
          save_status = f" | [Model Saved! Best Acc: {best_test_acc:.4f}]"
      else:
          save_status = ""

      # Вывод результатов
      print(f"Epoch {epoch:2d}/{EPOCHS} | "
            f"Train Loss: {train_loss_avg:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}{save_status}")


  print("=" * 70)
  print(f"ОБУЧЕНИЕ ЗАВЕРШЕНО. Лучшая точность: {best_test_acc:.4f}")
  print("=" * 70)

  # 9: Финальная оценка и детальные метрики
  # ==============================================

  from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

  # Получаем финальные предсказания


  # ЗАГРУЖАЕМ ЛУЧШИЕ ВЕСА ПЕРЕД ФИНАЛЬНОЙ ОЦЕНКОЙ
  if os.path.exists(tsmixer_output_dir):
      print("=" * 70)
      print(f"ЗАГРУЖАЕМ ЛУЧШИЕ ВЕСА ПЕРЕД ФИНАЛЬНОЙ ОЦЕНКОЙ")
      #model_binary.load_state_dict(torch.load(tsmixer_output_dir, weights_only=True))

      checkpoint = torch.load(tsmixer_output_dir, map_location=device)
      model_binary.load_state_dict(checkpoint['model_state_dict'])
      optimizer_binary = torch.optim.AdamW(model_binary.parameters(), lr=LR)
      if 'optimizer_state_dict' in checkpoint:
         optimizer_binary.load_state_dict(checkpoint['optimizer_state_dict'])
      model_binary.to(device)
      print(f"Загружены лучшие веса из {tsmixer_output_dir}")
  print("=" * 70)

  final_test_acc, y_pred_binary, y_true_binary = evaluate_binary_classifier()
  #torch.save(model_binary, tsmixer_output_dir)
  #print(f"Полная модель {tsmixer_output_dir} сохранена.")


  print("\n" + "=" * 70)
  print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ")
  print("=" * 70)
  print(f"Test Accuracy: {final_test_acc:.4f} ({final_test_acc*100:.2f}%)")

  # Baseline - всегда предсказываем мажоритарный класс
  baseline_acc = max(y_true_binary.mean(), 1 - y_true_binary.mean())
  print(f"Baseline (majority class): {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")
  print(f"Улучшение над baseline: {(final_test_acc - baseline_acc)*100:.2f}%")

  # Classification Report
  print("\n" + "-" * 70)
  print("CLASSIFICATION REPORT")
  print("-" * 70)
  print(classification_report(
      y_true_binary,
      y_pred_binary,
      target_names=['Down (0)', 'Up (1)'],
      digits=4
  ))

  # Confusion Matrix
  print("-" * 70)
  print("CONFUSION MATRIX")
  print("-" * 70)
  cm = confusion_matrix(y_true_binary, y_pred_binary)
  print("\n              Predicted")
  print("                0      1")
  print(f"Actual 0    {cm[0,0]:5d}  {cm[0,1]:5d}")
  print(f"       1    {cm[1,0]:5d}  {cm[1,1]:5d}")

  # Дополнительные метрики
  tn, fp, fn, tp = cm.ravel()
  precision_up = tp / (tp + fp) if (tp + fp) > 0 else 0
  recall_up = tp / (tp + fn) if (tp + fn) > 0 else 0
  precision_down = tn / (tn + fn) if (tn + fn) > 0 else 0
  recall_down = tn / (tn + fp) if (tn + fp) > 0 else 0

  print("\n" + "-" * 70)
  print("ИНТЕРПРЕТАЦИЯ")
  print("-" * 70)
  print(f"True Positives (правильно предсказан рост):     {tp}")
  print(f"True Negatives (правильно предсказано падение): {tn}")
  print(f"False Positives (ложный сигнал на рост):        {fp}")
  print(f"False Negatives (пропущенный рост):             {fn}")

  # 11: Бэктест торговых стратегий
  # ====================================

  # Рассчитаем реальные returns
  actual_returns_binary = np.zeros(len(y_true_binary))

  for i in range(len(idx_test_binary)):
      idx = idx_test_binary[i]
      if idx < len(df_binary):
          today_price = df_binary["Close"].iloc[idx]
          next_price = df_binary["y_next_close"].iloc[idx]
          actual_returns_binary[i] = (next_price - today_price) / today_price

  # Стратегия 1: Long only (покупаем только когда модель предсказывает рост)
  strategy_long_only = np.where(y_pred_binary == 1, actual_returns_binary, 0.0)

  # Стратегия 2: Long/Short (лонг при росте, шорт при падении)
  strategy_long_short = np.where(y_pred_binary == 1, actual_returns_binary, -actual_returns_binary)

  # Кумулятивные доходности
  cum_return_buy_hold = np.cumprod(1 + actual_returns_binary) - 1
  cum_return_long_only = np.cumprod(1 + strategy_long_only) - 1
  cum_return_long_short = np.cumprod(1 + strategy_long_short) - 1

  # Статистика
  print("\n" + "=" * 70)
  print("BACKTEST ТОРГОВЫХ СТРАТЕГИЙ")
  print("=" * 70)
  print(f"Buy & Hold:              {cum_return_buy_hold[-1]:7.2%}")
  print(f"Strategy (Long only):    {cum_return_long_only[-1]:7.2%}")
  print(f"Strategy (Long/Short):   {cum_return_long_short[-1]:7.2%}")
  print("=" * 70)

  # Дополнительные метрики
  sharpe_bh = actual_returns_binary.mean() / (actual_returns_binary.std() + 1e-8) * np.sqrt(252)
  sharpe_long = strategy_long_only.mean() / (strategy_long_only.std() + 1e-8) * np.sqrt(252)
  sharpe_ls = strategy_long_short.mean() / (strategy_long_short.std() + 1e-8) * np.sqrt(252)

  print(f"\nSharpe Ratio (годовой):")
  print(f"  Buy & Hold:           {sharpe_bh:.3f}")
  print(f"  Strategy (Long only): {sharpe_long:.3f}")
  print(f"  Strategy (Long/Short):{sharpe_ls:.3f}")
  print("=" * 70)

  # бэктест с помощью библиотеки
  test_datadf_neu = test_datadf_neu[ LOOKBACK: ]
  test_datadf_neu['Signal'] = y_pred_binary
  test_datadf_neu['Signal'] = test_datadf_neu['Signal'].ffill()
  test_datadf_neu_signals = test_datadf_neu['Signal']

  test_datadf_neu_backtest = setBktDataFrame(test_datadf_neu_signals, test_conditions_neu, test_datadf_neu)

  bt = Backtest(test_datadf_neu_backtest, ChronosNextDayDirection, cash=1000000, commission=.002, exclusive_orders=True, finalize_trades=True)

  stats_neu = bt.run()
  print(stats_neu[:27])
  bt.plot(
        plot_equity=True,
        plot_drawdown=True,
        relative_equity=False,
        superimpose=False #при date = 1h ошибка Invalid value for `superimpose`: Upsampling not supported
    )

  risk_adjusted_return_neu = (
            stats_neu.iloc[7] *
            stats_neu.iloc[12] *
            stats_neu.iloc[13] *
            stats_neu.iloc[7]
        ) / (1 + stats_neu.iloc[17])

  display(f'risk_adjusted_return = {risk_adjusted_return_neu:.2f}')

  # ROC AUC
  model_binary.eval()
  all_probs = []
  all_labels = []

  with torch.no_grad():
      for xb, yb in test_loader_binary:
          xb = xb.to(device)
          logits = model_binary(xb)
          # Получаем вероятности класса 1 (Up)
          probs = torch.softmax(logits, dim=1)[:, 1]

          all_probs.append(probs.cpu())
          all_labels.append(yb.cpu())

  # Объединяем все батчи в единые тензоры
  all_probs = torch.cat(all_probs)
  all_labels = torch.cat(all_labels)

  # Считаем метрику
  metric = BinaryAUROC()
  auc_score = metric(all_probs, all_labels)

  print(f"Финальный ROC AUC на тесте: {auc_score.item():.4f}")

  return_neu = {
            'final_equity': stats_neu.iloc[4], #final_equity,
            'n_trades': stats_neu.iloc[21], #n_trades,
            'returns': stats_neu.iloc[7], #returns,
            'max_drawdown': abs(stats_neu.iloc[17]), #max_drawdown,
            'win_rate': stats_neu.iloc[22], #win_rate,
            'sharpe_ratio': stats_neu.iloc[12], #sharpe_ratio,
            'sortino_ratio': stats_neu.iloc[13], #sortino_ratio,
            'profit_factor': stats_neu.iloc[7], #profit_factor
            'rocauc': auc_score, #rocauc для модели,
            'strategy_ticker': strategy.ticker
            #'pred': model_binary_pred.copy() # запоминаем предсказание для набора
        }
  #return_neu['final_equity'] = return_neu['final_equity'].map(lambda x: f"{x}")
  #return_neu['max_drawdown'] = return_neu['max_drawdown'].map(lambda x: x*100)

  return return_neu


In [ ]:
class TemaGoldenCrossStrategy(Strategy):
    def init(self):
        # переопределяем Индикатор по колонке Signal
        self.signal = self.I(lambda: self.data.Signal)
        self.previous_signal = 0

    def next(self):
        current_signal = self.signal[-1]

        ########################################################################
        #  previous_signal current_signal   previous_position    current_position
        #          0              1             is_short              close
        #          0              1        != is_long (close)      buy (is_long)
        #
        #         -1              1             is_short              close
        #         -1              1        != is_long (close)      buy (is_long)
        #
        #         (1)             1
        #
        if current_signal != self.previous_signal:
            if current_signal == 1:
                if self.position.is_short:
                    self.position.close()

                if not self.position.is_long:
                    self.buy()

        ########################################################################
        #  previous_signal current_signal   previous_position    current_position
        #          0              -1             is_long              close
        #          0              -1            != is_short            sell
        #
        #          1              -1             is_long              close
        #          1              -1            != is_short            sell
        #
        #        (-1)             -1

            elif current_signal == -1:
                if self.position.is_long:
                    self.position.close()

                if not self.position.is_short:
                    self.sell()

        ########################################################################
        #  previous_signal current_signal   previous_position    current_position
        #          1              0                                  close
        #
        #         -1              0                                  close
        #
        #          0              0                                  close

            elif current_signal == 0:
                if self.position:
                    self.position.close()

        self.previous_signal = current_signal

In [ ]:
def visualForBacktestML(train_df):

  good_signals = train_df[train_df['Label'] == 1].copy()
  bad_signals = train_df[train_df['Label'] == 0].copy()

  figure = make_subplots(rows=1, cols=1)

  # График цены закрытия и TEMA
  figure.add_trace(go.Scatter(x=train_df.index, y=train_df['Close'], name='Цена закрытия', line=dict(color='blue')), row=1, col=1)
  figure.add_trace(go.Scatter(x=good_signals.index, y=good_signals['Close'], mode='markers',
                            name='Покупка', marker=dict(symbol='triangle-up', size=10, color='green')), row=1, col=1)
  figure.add_trace(go.Scatter(x=bad_signals.index, y=bad_signals['Close'], mode='markers',
                            name='Продажа', marker=dict(symbol='triangle-down', size=10, color='red')), row=1, col=1)

  # Настройка макета графика
  figure.update_layout(height=800, title='Визуализация прибыльных и убыточных сделок')
  figure.update_yaxes(title_text='Цена', row=1, col=1)

  figure.show()

  return figure

Проверка работы стратегии с использованием формируемой модели машинного обучения и библиотеки backtesting

In [ ]:
#Проверка работы стратегии с использованием модели машинного обучения
def backtestWithML(strategy, data, test_data, ml_modelname=None, ml_model=None, model_pred=None):

    #train_df = signals.copy()
    train_signals, train_conditions, datadf = strategy.generate_signals(data)
    train_df = train_signals.copy()

    #train_conditions = conditions.copy()
    #bt_df = signals.copy()

    #print("Разделение на трейн и тест ")
    #split_value = int(len(bt_df) * 0.6)
    #train_df = bt_df.iloc[:split_value]
    ##
    #test_df = bt_df.iloc[:split_value]
    #test_df = bt_df.iloc[split_value:]#.reset_index(drop=True)



    bt_df = train_df.reset_index()
    bt_df.columns = ['Date', 'signal']
    bt_df['shift'] = bt_df['signal'].shift()
    bt_df['shift'] = bt_df['shift'].fillna(0)
    bt_df['action_x'] = bt_df['signal'].ne(bt_df['shift'])

    bt_df['action_id'] = 0
    bt_df['action_id'] = bt_df['signal'].ne(bt_df['signal'].shift()).cumsum()
    bt_df.columns = bt_df.columns.str.capitalize()

    data_backtest = datadf.copy()

    data_backtest.reset_index()
    data_backtest.drop('Signal', axis=1, inplace=True)
    data_merge = pd.merge(bt_df, data_backtest, on='Date', how='inner').dropna()

    data_merge.rename(columns={'Date': 'Datetime'}, inplace=True)
    data_merge["Datetime"] = pd.to_datetime(data_merge["Datetime"])
    data_merge.set_index('Datetime', inplace=True)
    data_merge.columns = data_merge.columns.str.capitalize()

    #Успешные и убыточные сделки
    # групируем по id оставляя только первые строки каждой сделки - это наши точки смены сигнала

    data_merge = data_merge.reset_index()
    train_df = data_merge.groupby('Action_id').first().reset_index()
    # добавим столбец с профитом
    train_df['profit'] = 0.0

    # Смещаем столбцы 'close' и 'signal', чтобы получить данные следующего бара
    train_df['next_close'] = train_df['Close'].shift(-1)
    train_df['next_signal'] = train_df['Signal'].shift(-1)

    # Рассчитываем PnL (отчет о прибылях и убытках)
    train_df['profit'] = (train_df['next_close'] - train_df['Close']) * train_df['Signal']

    # Заполняем PnL значением 0 для последнего бара, так как для него нет следующего бара
    train_df['profit'] = train_df['profit'].fillna(0)

    # Добавляем целевую переменную : 0 - cделка принесла убыток, 1 - сделка принесла прибыль
    train_df['Label'] = train_df['profit'].apply(lambda x: 1 if x >= 0 else 0)

    #!!!!!!2.2 Label = profit >= 0 - “нулевая прибыль” считается выигрышем
    #Это искусственно улучшает win-rate, особенно если много нулевых/почти нулевых движений.
    #Как лучше
    #- Label = profit > 0
    #- или event-based: profit > fee + slippage (т.е. прибыль должна перекрыть издержки)

    train_df = train_df[train_df['Signal'] != 0]
    train_df.set_index('Datetime', inplace=True)
    #train_df[['Close', 'next_close' , 'Signal', 'next_signal', 'profit', 'Label']]

    # Визуализация
    #visualForBacktestML(train_df)

    ##------------------

    #X_trainbkt, X_testbkt, y_trainbkt, y_testbkt = train_test_split(
    #train_df, train_df['Label'], test_size=0.3, random_state=42, stratify=train_df['Label']
    #)

    display("Создаем объект класса Backtest")

    X_train = train_df[['Signal', 'Open', 'High', 'Low', 'Close', 'Volume', 'Label']]
    #2.1 Label строится от будущей цены (shift(-1)) и затем используется как фича
    #Смотри у тебя:
    #python
    #train_df['next_close'] = train_df['Close'].shift(-1)
    #profit = (next_close - Close) * Signal
    #Label = 1 if profit >= 0 else 0
    #дальше формируется X_train = train_df[['Signal','Open','High','Low','Close','Volume','Label']] и делается Backtest(X_train, …)
    #Проблема (критично):
    #Label вычислен из будущего (через next_close), но ты кладёшь его в датафрейм, который потом подаётся в бэктест как входные данные. Даже если стратегия его напрямую не использует, это опасная утечка (и очень часто приводит к “магическим” результатам, особенно если где-то случайно используется колонка).
    #Как поправить - Никогда не держать Label, next_close, profit внутри датафрейма, который идёт в Backtest.
    #Делай отдельный df для обучения/оценки и отдельный OHLCV+Signal для бэктеста.

    # Создаем объект класса Backtest
    bt = Backtest(X_train, TemaGoldenCrossStrategy, cash=1_000_000, commission=.002, exclusive_orders=True, finalize_trades=True) # сделки идут последовательно


    display("Запускаем бэктест")

    # Запускаем бэктест
    stats = bt.run()

    ###bt.plot()

    db1 = stats[:27]

    # Выводим статистику
    #display("Выводим статистику")
    #print(stats[:27])

    ##------------------


    # И оставляем только те строки где мы входили в позицию
    #display("Оставляем только те строки где мы входили в позицию")
    #train_df = train_df[train_df['Signal'] != 0]

    # Удаляем вспомогательные столбцы
    train_df.drop(columns=['next_close', 'next_signal'], inplace=True)

    # Разделяем признаки и метки
    display("Разделяем признаки и метки")
    train_conditions.columns = train_conditions.columns.str.capitalize()
    train_conditions.drop('Signal', axis=1, inplace=True)

    X_train = train_df.filter(train_conditions.columns, axis=1)
    #X_train = train_df[['tema', 'macd', 'macd_signal', 'close', 'signal']]
    y_train = train_df['Label']


    display("RobustScaler")
    #scaler = StandardScaler()
    scaler = RobustScaler()
    x_scaled = scaler.fit_transform(X_train)
    #https://sky.pro/wiki/python/primer-ispolzovaniya-random-forest-classifier/

    # #обучаем модель
    #model = RandomForestClassifier(n_estimators=100, random_state=42)
    # # Определение гиперпараметров для настройки
    #param_grid = {
    #     'n_estimators': [50, 100, 200],
    #     'max_depth': [None, 10, 20, 30],
    #     'min_samples_split': [2, 5, 10],
    #     'min_samples_leaf': [1, 2, 4]
    # }
    # # Создание экземпляра GridSearchCV
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)
    # # Поиск лучших параметров
    #grid_search.fit(x_scaled, y_train)
    # # Вывод лучших параметров
    #print(f"Лучшие параметры: {grid_search.best_params_}")
    # # Создание модели с лучшими параметрами
    #model = grid_search.best_estimator_
    #model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth = None, min_samples_leaf = 4, min_samples_split=2)

    #обучаем модель
    #model = RandomForestClassifier(random_state=42)
    model = CatBoostClassifier(random_state=42)
    model.fit(x_scaled, y_train,
              verbose = False,
              plot=False)
    if ml_modelname is not None :
      namemodel = ml_modelname
      model = joblib.load(namemodel)


    # Важность признаков
    feature_importance = pd.Series(model.feature_importances_, index=X_train.columns)
    print("Feature Importance:\n", feature_importance.sort_values(ascending=False))

    # повторяем все тоже самое с данными для теста
    display("Повторяем все тоже самое с данными для теста")

    test_signals, test_conditions, test_df = strategy.generate_signals(test_data)
    visual_signals = test_signals.copy()
    visual_signals = visual_signals.fillna(0)
    visual_signals = visual_signals.reindex(test_data.index, fill_value=0)
    close_prices = test_data['Close'].values
    if test_signals is None or test_signals.empty:
      return None
    test_signals = test_signals.fillna(0)
    test_signals = test_signals.reindex(test_data.index, fill_value=0)
    test_signals = test_signals.reset_index()
    test_signals.columns = ['Date', 'signal']
    test_df.drop('Signal', axis=1, inplace=True)
    test_df = pd.merge(test_signals, test_df, on='Date', how='inner').dropna()
    test_df.rename(columns={'Date': 'Datetime'}, inplace=True)
    test_df["Datetime"] = pd.to_datetime(test_df["Datetime"])
    test_df.set_index('Datetime', inplace=True)
    test_df['shift'] = test_df['signal'].shift()
    test_df['shift'] = test_df['shift'].fillna(0)
    test_df['action_x'] = test_df['signal'].ne(test_df['shift'])
    test_df['action_id'] = 0
    test_df['action_id'] = test_df['signal'].ne(test_df['signal'].shift()).cumsum()
    test_df = test_df.reset_index()
    test_df = test_df.groupby('action_id').first().reset_index()
    test_df['next_close'] = test_df['Close'].shift(-1)
    test_df['next_signal'] = test_df['signal'].shift(-1)
    test_df['profit'] = (test_df['next_close'] - test_df['Close']) * test_df['signal']
    test_df['profit'] = test_df['profit'].fillna(0)
    test_df['Label'] = test_df['profit'].apply(lambda x: 1 if x >= 0 else 0)

    #!!!!!!2.2 Label = profit >= 0 - “нулевая прибыль” считается выигрышем
    #Это искусственно улучшает win-rate, особенно если много нулевых/почти нулевых движений.
    #Как лучше
    #- Label = profit > 0
    #- или event-based: profit > fee + slippage (т.е. прибыль должна перекрыть издержки)
    #test_df = test_df[test_df['signal'] != 0]

    test_df.set_index('Datetime', inplace=True)
    test_df.columns = test_df.columns.str.capitalize()


    display("Для предсказания используем данные в столбацах ниже")
    # Для предсказания используем данные в столбацах condition
    #X_test = test_df[['tema', 'macd', 'macd_signal', 'close', 'signal']]
    test_conditions.columns = test_conditions.columns.str.capitalize()
    test_conditions.drop('Signal', axis=1, inplace=True)
    X_test = test_df.filter(test_conditions.columns, axis=1)
    y_test = test_df['Label']

    # делаем предсказание
    y_pred = model.predict(X_test)
    # вариант когда у нас уже есть модель
    if model_pred is not None:
      y_pred = model_pred.copy()

    # описание наших метрик
    # accuracy - точность нашей модели
    # precision - точность предсказания
    # recall - полнота предсказания
    # f1-score - среднее гармоническое точности и полноты
    # support - количество образцов в каждом классе
    # выводим предсказание
    display("Выводим предсказание")
    print(y_pred)
    # выведем нашу точность
    print(accuracy_score(y_test, y_pred))
    # выведем нашу матрицу
    print(confusion_matrix(y_test, y_pred))
    # выведем нашу классификационную отчет
    print(classification_report(y_test, y_pred))

    # Подготовим данные для бэктеста
    test_df['pred'] = y_pred
    test_df['pred'] = test_df['pred'].ffill()
    test_df.columns = test_df.columns.str.capitalize()
    test_df.loc[(test_df['Pred'] == 0) & (test_df['Signal'] != 0), 'Signal'] = 0
    # зануляем сделки предсказанные как убыточные
    #bt_df.loc[(bt_df['pred'] == 0) & (bt_df['Signal'] != 0), 'Signal'] = 0


    bt = Backtest(test_df, TemaGoldenCrossStrategy, cash=1000000, commission=.002, exclusive_orders=True, finalize_trades=True)

    # Запускаем бэктест
    stats = bt.run()

    # Выводим статистику
    print(stats[:27])

    display("Выводим график :")
    # выводим график

    bt.plot(
        plot_equity=True,
        plot_drawdown=True,
        relative_equity=False,
        superimpose=False #при date = 1h ошибка Invalid value for `superimpose`: Upsampling not supported
    )

    #backtest_strategy(strategy_params, test_df)

    display("Выводим Buy&Hold и GoldenCross :")

    db2 = stats[:27]
    db = pd.concat([db1, db2], axis = 1)
    db.columns = ['Buy&Hold', 'GoldenCross']
    display(db)

   # match = re.search(r"\d+\.\d+", db2.iloc[4])
   # if match:
   #   float_number = np.float(match.group(1))
   #   print(float_number)

    risk_adjusted_return_ml = (
            db2.iloc[7] *
            db2.iloc[12] *
            db2.iloc[13] *
            db2.iloc[7]
        ) / (1 + db2.iloc[17])

    print(f"risk_adjusted_return = {risk_adjusted_return_ml:.2f}")

    # Попытка отрисовки временного ряда с помощью не backtesting
    position_open = False
    positions = []
    equity = 1.0
    equity_curve = [equity]

    for i in range(1, len(visual_signals)):
            current_signal = visual_signals.iloc[i]
            current_date = visual_signals.index[i]

            if not position_open and current_signal != 0:
                # Открываем позицию
                position_open = True
                entry_price = close_prices[i]
                entry_signal = current_signal
                entry_date = current_date

            elif position_open and ((current_signal != 0 and current_signal != entry_signal) or i == len(visual_signals)-1):
                # Закрываем позицию
                exit_price = close_prices[i]
                profit = (exit_price - entry_price) / entry_price * entry_signal

                positions.append({
                    'entry_date': entry_date,
                    'exit_date': current_date,
                    'profit': float(profit)
                })
                equity *= (1.0 + profit)
                position_open = False

            equity_curve.append(float(equity))
    # Попытка отрисовки временного ряда с помощью не backtesting

    strategy.positions = positions
    strategy.equity = equity_curve
    strategy.signals = visual_signals

    roc = rocauc(model, X_test, y_test)

    return_ml = {
            'final_equity': db2.iloc[4], #final_equity,
            'n_trades': db2.iloc[21], #n_trades,
            'returns': db2.iloc[7], #returns,
            'max_drawdown': abs(db2.iloc[17]), #max_drawdown,
            'win_rate': db2.iloc[22], #win_rate,
            'sharpe_ratio': db2.iloc[12], #sharpe_ratio,
            'sortino_ratio': db2.iloc[13], #sortino_ratio,
            'profit_factor': db2.iloc[7], #profit_factor
            'rocauc': roc, #rocauc для модели
            'pred': y_pred.copy(), # запоминаем предсказание для набора
            'ml_model': joblib.dump(model, 'random_forest.joblib' + str(roc)),
            'ml_modelname': 'random_forest.joblib' + str(roc)
        }

    return return_ml


In [ ]:
# Класс для базовой стратегии
class BaseStrategy:
    def __init__(self):
        self.positions = []
        self.equity = []
        self.ticker = None

    def calculate_metrics(self, data):
        """Описание используемых метрик производительности:
           final_equity: Конечный капитал после завершения торгового периода. Показывает, насколько вырос или уменьшился начальный капитал в результате торговли.
           Используется для оценки общей эффективности стратегии. Чем выше конечный капитал, тем успешнее стратегия.
           n_trades: Количество сделок, совершенных за период тестирования. Показывает активность стратегии. Большое количество сделок может указывать на высокую частоту торговли, что может быть как преимуществом, так и недостатком в зависимости от контекста.
           returns: Общая доходность стратегии в процентах за период тестирования. Основной показатель прибыльности стратегии. Положительная доходность указывает на успешность стратегии, отрицательная — на убыточность.
           max_drawdown: Максимальная просадка капитала за период тестирования. Показывает наибольшее падение капитала от пика до минимума. Используется для оценки риска стратегии. Чем меньше просадка, тем стабильнее стратегия.
           win_rate: Процент прибыльных сделок от общего количества сделок. Показывает, насколько часто стратегия приносит прибыль. Высокий процент выигрышных сделок может указывать на стабильность стратегии.
           sharpe_ratio: Коэффициент Шарпа измеряет доходность с поправкой на риск. Чем выше значение, тем лучше стратегия с точки зрения доходности на единицу риска. Используется для сравнения стратегий с учетом риска. Высокий коэффициент Шарпа указывает на эффективное использование риска.
           sortino_ratio: Коэффициент Сортино, который измеряет доходность с поправкой на риск, учитывая только негативную волатильность. Аналогичен коэффициенту Шарпа, но фокусируется на риске убытков, что делает его более подходящим для оценки стратегий с асимметричным распределением доходности.
           profit_factor: Отношение общей прибыли к общему убытку. Показывает, насколько прибыльна стратегия по сравнению с убытками. Используется для оценки соотношения прибыли и убытков. Значение больше 1 указывает на прибыльность стратегии.
        """
        if not self.positions:
            return {
                'final_equity': 1.0,
                'n_trades': 0,
                'returns': 0,
                'max_drawdown': 0,
                'win_rate': 0,
                'sharpe_ratio': 0,
                'sortino_ratio': 0,
                'profit_factor': 0
            }

        # Расчет базовых метрик
        final_equity = self.equity[-1]
        n_trades = len(self.positions)
        returns = (final_equity - 1.0) * 100

        # Расчет максимальной просадки
        peak = 1.0
        drawdowns = []
        for eq in self.equity:
            if eq > peak:
                peak = eq
            drawdown = (peak - eq) / peak * 100
            drawdowns.append(drawdown)
        max_drawdown = max(drawdowns)

        # Расчет win rate
        profitable_trades = sum(1 for pos in self.positions if pos['profit'] > 0)
        win_rate = profitable_trades / n_trades * 100 if n_trades > 0 else 0

        # Расчет Sharpe и Sortino ratio
        daily_returns = pd.Series(self.equity).pct_change().dropna()
        if len(daily_returns) > 0:
            sharpe_ratio = np.sqrt(252) * daily_returns.mean() / daily_returns.std()
            downside_returns = daily_returns[daily_returns < 0]
            sortino_ratio = np.sqrt(252) * daily_returns.mean() / downside_returns.std() if len(downside_returns) > 0 else 0
        else:
            sharpe_ratio = sortino_ratio = 0

        # Расчет profit factor
        gross_profit = sum(pos['profit'] for pos in self.positions if pos['profit'] > 0)
        gross_loss = abs(sum(pos['profit'] for pos in self.positions if pos['profit'] < 0))
        profit_factor = gross_profit / gross_loss if gross_loss != 0 else 0

        return {
            'final_equity': final_equity,
            'n_trades': n_trades,
            'returns': returns,
            'max_drawdown': max_drawdown,
            'win_rate': win_rate,
            'sharpe_ratio': sharpe_ratio,
            'sortino_ratio': sortino_ratio,
            'profit_factor': profit_factor
        }

In [ ]:
# Стратегия 4: Золотой крест
#https://sergeytereshkin.ru/library/zolotoy-krest-signal-razvorota-i-metodika-poiska-na-grafike
class Strategy4(BaseStrategy):
    """
    «Золотой крест» — один из самых надежных бычьих сигналов технического анализа,
    позволяющий трейдерам структурировать вход в длинную позицию при строгом риск-менеджменте и подтверждении индикаторами.

    Выбор и настройка скользящих средних
    Оптимальные периоды для SMA
     - Для дневных и недельных графиков оптимальным сочетанием является SMA50/SMA200, поскольку SMA сглаживает шум и отражает среднесрочную и долгосрочную динамику цены.

    Использование EMA на коротких таймфреймах
     - На коротких таймфреймах (M15–H1) предпочтительнее EMA50/EMA200: EMA быстрее реагирует на последние изменения цены, что позволяет оперативнее фиксировать разворот, но требует дополнительной фильтрации шума.

    Бэктестинг настроек MA
     - Оптимальные параметры следует подбирать через бэктестинг: тестировать разные комбинации периодов,
       фиксируя доходность и максимальную просадку каждой стратегии.

    Подтверждающие индикаторы и фильтры
     - MACD как индикатор тренда
     - MACD подтверждает силу бычьего сигнала при росте гистограммы выше нулевой линии и пересечении сигнальной линии снизу вверх,
       что усиливает вероятность продолжения восходящего движения.

    RSI и оценка перекупленности
     - RSI в диапазоне 50–70 указывает на адекватный бычий импульс без перегрева рынка и помогает отсеивать ложные сигналы «золотого креста» во время флета.

    Анализ объемов и свечных паттернов
     - Дополнительным фильтром служат объемы: рост торгового объема в момент пересечения показывает активное участие игроков и подтверждает значимость сигнала.
       Бычьи свечные формации, такие как поглощение или «утренняя звезда», рядом с точкой пересечения усиливают доверие к сигналу.

    Источник Yandex:
    Некоторые оптимальные параметры стратегии Golden Cross в трейдинге:
      - Первичные условия: пересечение MA(50) над MA(200) с закреплением выше на 2+ торговых дня.
      - Направленность скользящих средних: обе MA направлены вверх под углом не менее 5°.
      - Расстояние между MA: минимальный разрыв 1,5% для исключения случайных пересечений.
      - Объёмное подтверждение: превышение 20-дневного среднего объёма на 30%+.
      - Дополнительные фильтры качества:
         RSI(14) в диапазоне 45–70 (для исключения экстремальных состояний перекупленности),
         MACD выше нулевой линии (для подтверждения смены импульса на бычий),
         отсутствие сильных уровней сопротивления в ближайших 8–12% от цены пересечения,
         макроэкономический фон (отсутствие ожидаемых негативных событий в ближайшие 4–6 недель).

    """
    def __init__(self, bb_period=20, bb_std=2, rsi_period=14, sma=10, ema_period=4, ma_period=10, macd_fast=12, macd_slow=26, macd_signal=9, stoch_k=14, stoch_d=3, stoch_slow=3):
        super().__init__()
        self.bb_period = int(bb_period)
        self.bb_std = float(bb_std)
        self.rsi_period = int(rsi_period)
        self.sma = int(sma)
        self.ema_period = int(ema_period)
        self.ma_period = int(ma_period)
        self.macd_fast = int(macd_fast)
        self.macd_slow = int(macd_slow)
        self.macd_signal = int(macd_signal)
        self.stoch_k = int(stoch_k)
        self.stoch_d = int(stoch_d)
        self.stoch_slow = int(stoch_slow)

    def get_params(self):
        """Возвращает текущие гиперпараметры стратегии"""
        return {
            'bb_period': self.bb_period,
            'bb_std': self.bb_std,
            'rsi_period': self.rsi_period,
            'sma': self.sma,
            'ema_period': self.ema_period,
            'ma_period': self.ma_period,
            'macd_fast': self.macd_fast,
            'macd_slow': self.macd_slow,
            'macd_signal': self.macd_signal,
            'stoch_k': self.stoch_k,
            'stoch_d': self.stoch_d,
            'stoch_slow': self.stoch_slow,
        }

    def generate_signals(self, data):
        try:
            df = data.copy()

            close_prices = df['Close'].values.astype(float).flatten()
            open_prices = df['Open'].values.astype(float).flatten()
            high_prices = df['High'].values.astype(float).flatten()
            low_prices = df['Low'].values.astype(float).flatten()
            volume = df['Volume'].values.astype(float).flatten()

            #Полосы Боллинджера - Уровни поддержки и сопротивления
            df['BB_middle'] = ta.SMA(close_prices, timeperiod=self.bb_period)
            std_dev = ta.STDDEV(close_prices, timeperiod=self.bb_period)
            df['BB_upper'] = df['BB_middle'] + self.bb_std * std_dev
            df['BB_lower'] = df['BB_middle'] - self.bb_std * std_dev

            df['sma'] = ta.SMA(close_prices, timeperiod=self.sma)
            df['ma_period'] = ta.MA(close_prices, timeperiod=self.ma_period)
            df['RSI'] = ta.RSI(close_prices, timeperiod=self.rsi_period)
            #df['obv'] = np.where(df['Close'] > df['Close'].shift(1), df['Volume'], np.where(df['Close'] < df['Close'].shift(1), -df['Volume'], 0)).cumsum()
            df['obv'] = ta.OBV(close_prices, volume)
            df['engulfing'] = ta.CDLENGULFING(open_prices, high_prices, low_prices, close_prices)
            df['morningstar'] = ta.CDLMORNINGSTAR(open_prices, high_prices, low_prices, close_prices)
            df['adosc'] = ta.ADOSC(high_prices, low_prices, close_prices, volume)
            slowk, slowd = ta.STOCH(high_prices,
                                   low_prices,
                                   close_prices,
                                   fastk_period=self.stoch_k,
                                   slowk_period=self.stoch_d,
                                   slowd_period=self.stoch_slow)
            df['slowk'] = slowk
            df['slowd'] = slowd
            df['doji'] = ta.CDLDOJI(open_prices, high_prices, low_prices, close_prices)
            df['hammer'] = ta.CDLHAMMER(open_prices, high_prices, low_prices, close_prices)
            df['atr14'] = ta.ATR(high_prices, low_prices, close_prices, timeperiod=14)
            df['sar'] = ta.SAR(high_prices, low_prices, acceleration=0, maximum=0)
            df['cci'] = ta.CCI(high_prices, low_prices, close_prices, timeperiod=14)
            df['adx'] = ta.ADX(high_prices, low_prices, close_prices, timeperiod=14)
            df['mfi'] = ta.MFI(high_prices, low_prices, close_prices, volume, timeperiod=14)
            df['linearreg_angle'] = ta.LINEARREG_ANGLE(close_prices, timeperiod=14)

            macd, signal, _ = ta.MACD(close_prices,
                              fastperiod=self.macd_fast,
                              slowperiod=self.macd_slow,
                              signalperiod=self.macd_signal)

            df['MACD'] = macd
            df['Signal'] = signal
            #df = df.fillna(method='ffill').fillna(method='bfill')
            df = df.ffill().copy()
            signals = pd.Series(index=df.index, data=0)

            long_condition = ((df['engulfing'] > 0) | (df['morningstar'] > 0)) #& ((df['Close'] < df['sma']) & (df['Close'].shift(1) >= df['sma'].shift(1))) & (df['MACD'] > df['Signal']) & (df['RSI'] > 50) & (df['RSI'] < 70) & (df['obv'] > 0)
            #long_condition =  (df['RSI'] > 50) & (df['sma_period'] > 50) & (df['Close'] > df['EMA']) & (df['MACD'] > df['Signal']) & (df['obv'] > 0)
            #short_condition =  (df['RSI'] < 70) & (df['sma_period'] < 200) & (df['Close'] < df['EMA']) & (df['MACD'] < df['Signal']) & (df['obv'] < 0)
            #short_condition = Стоп-лосс | тейк-профит | трейлинг-стоп
            short_condition =  (((df['sma'] < 50)) | ((df['Close'] < df['sma']) & (df['Close'].shift(1) >= df['sma'].shift(1)))) | ((df['RSI'] > 50) & (df['MACD'] < df['Signal'])) | (df['Close'] > df['BB_upper'])
            signals[long_condition] = 1
            signals[short_condition] = -1
            signals = signals.reindex_like(data)


            conditions = df.copy()
            conditions = conditions.reset_index()
            conditions = df.filter(['BB_middle', 'BB_upper',
                                    'BB_lower', 'sma', 'ma_period', 'RSI', 'obv', 'engulfing',
                                    'morningstar', 'MACD', 'slowk', 'slowd', 'doji', 'hammer', 'atr14', 'sar',
                                    'adosc', 'linearreg_angle', 'cci','adx','mfi', 'Signal'], axis=1)
            conditions = conditions.dropna()

            return signals, conditions, df
        except Exception as e:
            print(f"Ошибка в generate_signals для Strategy4: {str(e)}")
            return None

In [ ]:
# Стратегия 3: RSI + Candlestick Patterns
class Strategy3(BaseStrategy):
    """
    Комбинированная стратегия разворота тренда, использующая:
    - RSI (Relative Strength Index) - для определения перекупленности/перепроданности
    - Паттерны свечей - для подтверждения разворота (Doji, Hammer, Engulfing)

    Бизнес-логика стратегии:
    1. Основной принцип:
       - Поиск точек разворота в перекупленных/перепроданных зонах
       - Использование паттернов свечей для подтверждения разворота

    2. Условия для входа в длинную позицию (Long):
       - RSI находится в зоне перепроданности (< 30)
       - Формируется один из бычьих паттернов:
         * Doji (свеча неопределенности)
         * Hammer (молот)
         * Bullish Engulfing (бычье поглощение)

    3. Условия для входа в короткую позицию (Short):
       - RSI находится в зоне перекупленности (> 70)
       - Формируется один из медвежьих паттернов:
         * Doji (свеча неопределенности)
         * Inverted Hammer (перевернутый молот)
         * Bearish Engulfing (медвежье поглощение)

    4. Управление рисками:
       - RSI фильтрует слабые сигналы
       - Требуется подтверждение паттернами свечей
       - Учитывается сила сигнала разворота

    Особенности настройки:
    - rsi_period: период RSI (по умолчанию 14)
    - Паттерны свечей имеют фиксированные правила идентификации

    Преимущества стратегии:
    - Сочетание технического индикатора и визуального анализа
    - Четкие правила входа и выхода
    - Возможность раннего обнаружения разворотов тренда
    """
    def __init__(self, rsi_period=14):
        super().__init__()
        self.rsi_period = int(rsi_period)

    def get_params(self):
        """Возвращает текущие гиперпараметры стратегии"""
        return {
            'rsi_period': self.rsi_period
        }

    def generate_signals(self, data):
        try:
            df = data.copy()
            close_prices = df['Close'].values.astype(float).flatten()
            open_prices = df['Open'].values.astype(float).flatten()
            high_prices = df['High'].values.astype(float).flatten()
            low_prices = df['Low'].values.astype(float).flatten()
            df['RSI'] = ta.RSI(close_prices, timeperiod=self.rsi_period)
            df['doji'] = ta.CDLDOJI(open_prices, high_prices, low_prices, close_prices)
            df['hammer'] = ta.CDLHAMMER(open_prices, high_prices, low_prices, close_prices)
            df['engulfing'] = ta.CDLENGULFING(open_prices, high_prices, low_prices, close_prices)
            df = df.ffill().copy()
            #df = df.fillna(method='ffill').fillna(method='bfill')
            signals = pd.Series(index=df.index, data=0)
            long_condition = ((df['doji'] > 0) | (df['hammer'] > 0) | (df['engulfing'] > 0)) & (df['RSI'] < 30)
            short_condition = ((df['doji'] < 0) | (df['hammer'] < 0) | (df['engulfing'] < 0)) & (df['RSI'] > 70)
            signals[long_condition] = 1
            signals[short_condition] = -1
            signals = signals.reindex_like(data)

            conditions = df.copy()
            conditions = conditions.reset_index()
            conditions = df.filter(['RSI', 'doji',
                                    'hammer', 'engulfing'], axis=1)
            conditions = conditions.dropna()

            return signals, conditions, df
        except Exception as e:
            print(f"Ошибка в generate_signals для Strategy3: {str(e)}")
            return None

In [ ]:
class Strategy2(BaseStrategy):
    """
    Стратегия торговли на отскок (Mean Reversion), использующая:
    - Bollinger Bands - для определения волатильности и экстремальных значений цены
    - Stochastic Oscillator - для подтверждения разворота

    Бизнес-логика стратегии:
    1. Основной принцип:
       - Торговля от краев полос Боллинджера к средней линии
       - Использование стохастика для подтверждения сигналов разворота

    2. Условия для входа в длинную позицию (Long):
       - Цена находится ниже нижней полосы Боллинджера (перепроданность)
       - Линия %K стохастика пересекает линию %D снизу вверх (подтверждение разворота вверх)

    3. Условия для входа в короткую позицию (Short):
       - Цена находится выше верхней полосы Боллинджера (перекупленность)
       - Линия %K стохастика пересекает линию %D сверху вниз (подтверждение разворота вниз)

    4. Управление рисками:
       - Полосы Боллинджера автоматически адаптируются к волатильности
       - Стохастик подтверждает потенциал разворота
       - Торговля против тренда только при сильных сигналах

    Особенности настройки:
    - bb_period: период Bollinger Bands (по умолчанию 20)
    - bb_std: количество стандартных отклонений (по умолчанию 2)
    - stoch_k/d/slow: параметры стохастика, влияющие на его чувствительность
    """
    def __init__(self, bb_period=20, bb_std=2, stoch_k=14, stoch_d=3, stoch_slow=3, rsi_period = 14):
        super().__init__()
        self.bb_period = int(bb_period)
        self.bb_std = float(bb_std)
        self.stoch_k = int(stoch_k)
        self.stoch_d = int(stoch_d)
        self.stoch_slow = int(stoch_slow)
        self.rsi_period = int(rsi_period)

    def get_params(self):
        """Возвращает текущие гиперпараметры стратегии"""
        return {
            'bb_period': self.bb_period,
            'bb_std': self.bb_std,
            'stoch_k': self.stoch_k,
            'stoch_d': self.stoch_d,
            'stoch_slow': self.stoch_slow,
            'rsi_period': self.rsi_period
        }

    def generate_signals(self, data):
        try:
            df = data.copy()
            close_prices = df['Close'].values.astype(float).flatten()
            high_prices = df['High'].values.astype(float).flatten()
            low_prices = df['Low'].values.astype(float).flatten()
            df['BB_middle'] = ta.SMA(close_prices, timeperiod=self.bb_period)
            std_dev = ta.STDDEV(close_prices, timeperiod=self.bb_period)
            df['BB_upper'] = df['BB_middle'] + self.bb_std * std_dev
            df['BB_lower'] = df['BB_middle'] - self.bb_std * std_dev
            df['RSI'] = ta.RSI(close_prices, timeperiod=self.rsi_period)
            slowk, slowd = ta.STOCH(high_prices,
                                   low_prices,
                                   close_prices,
                                   fastk_period=self.stoch_k,
                                   slowk_period=self.stoch_d,
                                   slowd_period=self.stoch_slow)
            df['slowk'] = slowk
            df['slowd'] = slowd
            df = df.ffill().copy()
            #df = df.fillna(method='ffill').fillna(method='bfill')
            signals = pd.Series(index=df.index, data=0)
            long_condition = (df['Close'] < df['BB_lower']) & (df['slowk'] > df['slowd']) & (df['RSI'] < 70)
            short_condition = (df['Close'] > df['BB_upper']) & (df['slowk'] < df['slowd']) & (df['RSI'] > 30)
            signals[long_condition] = 1
            signals[short_condition] = -1
            signals = signals.reindex_like(data)

            conditions = df.copy()
            conditions = conditions.reset_index()
            conditions = df.filter(['BB_middle', 'BB_upper',
                                    'BB_lower', 'RSI', 'slowk', 'slowd', 'Close'], axis=1)
            conditions = conditions.dropna()

            return signals, conditions, df
        except Exception as e:
            print(f"Ошибка в generate_signals для Strategy2: {str(e)}")
            return None




In [ ]:
class Strategy1(BaseStrategy):
    """
    Комбинированная трендово-импульсная стратегия, использующая:
    - EMA (Exponential Moving Average) - для определения тренда
    - RSI (Relative Strength Index) - для определения перекупленности/перепроданности
    - MACD (Moving Average Convergence Divergence) - для определения моментума и точек входа/выхода

    Бизнес-логика стратегии:
    1. Определение тренда:
       - Цена выше EMA = восходящий тренд
       - Цена ниже EMA = нисходящий тренд

    2. Условия для входа в длинную позицию (Long):
       - Цена находится выше EMA (подтверждение восходящего тренда)
       - RSI < 70 (нет перекупленности)
       - MACD пересекает Signal Line снизу вверх (сигнал усиления восходящего импульса)

    3. Условия для входа в короткую позицию (Short):
       - Цена находится ниже EMA (подтверждение нисходящего тренда)
       - RSI > 30 (нет перепроданности)
       - MACD пересекает Signal Line сверху вниз (сигнал усиления нисходящего импульса)

    4. Управление рисками:
       - RSI используется как фильтр для предотвращения входа в перекупленной/перепроданной зоне
       - EMA подтверждает направление тренда
       - MACD подтверждает силу импульса

    Особенности настройки:
    - ema_period: период EMA (по умолчанию 20) - чем больше период, тем более долгосрочный тренд отслеживается
    - rsi_period: период RSI (по умолчанию 14) - влияет на чувствительность индикатора к изменениям цены
    - macd_fast/slow/signal: периоды MACD - влияют на скорость реакции на изменение тренда
    """
    def __init__(self, ema_period=20, rsi_period=14, macd_fast=12, macd_slow=26, macd_signal=9):
        super().__init__()
        self.ema_period = int(ema_period)
        self.rsi_period = int(rsi_period)
        self.macd_fast = int(macd_fast)
        self.macd_slow = int(macd_slow)
        self.macd_signal = int(macd_signal)

    def get_params(self):
        """Возвращает текущие гиперпараметры стратегии"""
        return {
            'ema_period': self.ema_period,
            'rsi_period': self.rsi_period,
            'macd_fast': self.macd_fast,
            'macd_slow': self.macd_slow,
            'macd_signal': self.macd_signal
        }

    def generate_signals(self, data):
        try:
            df = data.copy()
            close_prices = df['Close'].values.astype(float).flatten()
            df['EMA'] = ta.EMA(close_prices, timeperiod=self.ema_period)
            df['RSI'] = ta.RSI(close_prices, timeperiod=self.rsi_period)
            macd, signal, _ = ta.MACD(close_prices,
                                      fastperiod=self.macd_fast,
                                      slowperiod=self.macd_slow,
                                      signalperiod=self.macd_signal)
            df['MACD'] = macd
            df['Signal'] = signal
            df = df.ffill().copy()
            #df = df.fillna(method='ffill').fillna(method='bfill')
            signals = pd.Series(index=df.index, data=0)
            long_condition = (df['Close'] > df['EMA']) & (df['RSI'] < 70) & (df['MACD'] > df['Signal'])
            short_condition = (df['Close'] < df['EMA']) & (df['RSI'] > 30) & (df['MACD'] < df['Signal'])
            signals[long_condition] = 1
            signals[short_condition] = -1
            signals = signals.reindex_like(data)

            conditions = df.copy()
            conditions = conditions.reset_index()
            conditions = df.filter(['EMA', 'RSI',
                                    'MACD', 'Signal', 'Close'], axis=1)
            conditions = conditions.dropna()

            return signals, conditions, df
        except Exception as e:
            print(f"Ошибка в generate_signals для Strategy1: {str(e)}")
            return None


In [ ]:
# Определение пространств параметров для оптимизации
param_spaces = {
    'Strategy1': {
        'ema_period': hp.quniform('ema_period', 10, 50, 1),
        'rsi_period': hp.quniform('rsi_period', 7, 21, 1),
        'macd_fast': hp.quniform('macd_fast', 8, 20, 1),
        'macd_slow': hp.quniform('macd_slow', 21, 40, 1),
        'macd_signal': hp.quniform('macd_signal', 5, 15, 1)
    },
    'Strategy2': {
        'bb_period': hp.quniform('bb_period', 10, 50, 1),
        'bb_std': hp.uniform('bb_std', 1.5, 3.0),
        'stoch_k': hp.quniform('stoch_k', 7, 21, 1),
        'stoch_d': hp.quniform('stoch_d', 3, 7, 1),
        'stoch_slow': hp.quniform('stoch_slow', 3, 7, 1),
        'rsi_period': hp.quniform('rsi_period', 7, 21, 1)
    },
    'Strategy3': {
        'rsi_period': hp.quniform('rsi_period', 7, 21, 1)
    },
    'Strategy4': {
        'bb_period': hp.quniform('bb_period', 10, 50, 1),
        'bb_std': hp.uniform('bb_std', 1.5, 3.0),
        'rsi_period': hp.quniform('rsi_period', 7, 21, 1),
        'sma': hp.quniform('sma', 10, 50, 1),
        'ema_period': hp.quniform('ema_period', 10, 50, 1),
        'ma_period': hp.quniform('ma_period', 10, 50, 1),
        'macd_fast': hp.quniform('macd_fast', 8, 20, 1),
        'macd_slow': hp.quniform('macd_slow', 21, 40, 1),
        'macd_signal': hp.quniform('macd_signal', 5, 15, 1),
        'stoch_k': hp.quniform('stoch_k', 7, 21, 1),
        'stoch_d': hp.quniform('stoch_d', 3, 7, 1),
        'stoch_slow': hp.quniform('stoch_slow', 3, 7, 1),
    }
}

In [ ]:
# Функция для бэктестирования Buy & Hold стратегии
def backtest_buy_hold(data):
    """Бэктестирование Buy & Hold стратегии"""
    initial_price = float(data['Close'].iloc[0])
    final_price = float(data['Close'].iloc[-1])
    returns = (final_price - initial_price) / initial_price * 100

    daily_returns = data['Close'].pct_change().dropna()

    if len(daily_returns) > 0:
        sharpe_ratio = np.sqrt(252) * daily_returns.mean() / daily_returns.std()
        downside_returns = daily_returns[daily_returns < 0]
        sortino_ratio = np.sqrt(252) * daily_returns.mean() / downside_returns.std() if len(downside_returns) > 0 else 0
    else:
        sharpe_ratio = sortino_ratio = 0

    rolling_max = data['Close'].expanding().max()
    drawdowns = (rolling_max - data['Close']) / rolling_max
    max_drawdown = float(drawdowns.max() * 100)

    return {
        'final_equity': 1 + returns / 100,
        'n_trades': 1,
        'returns': returns,
        'max_drawdown': max_drawdown,
        'win_rate': 100.0 if returns > 0 else 0.0,
        'sharpe_ratio': float(sharpe_ratio),
        'sortino_ratio': float(sortino_ratio),
        'profit_factor': float('inf') if returns > 0 else 0.0
    }


In [ ]:
def optimize_strategy(strategy_class, data, param_space):
    def objective(params):
        strategy = strategy_class(**params)
        metrics, conditions, signals, df = backtest_strategy(strategy, data)
        if metrics:
            return {'loss': -metrics['profit_factor'], 'status': STATUS_OK}
        else:
            return {'loss': 0, 'status': STATUS_OK}

    trials = Trials()
    best = fmin(fn=objective,
                space=param_space,
                algo=tpe.suggest,
                max_evals=50,
                trials=trials)

    return best

In [ ]:
def format_metric(value):
    """Форматирует число с двумя знаками после точки, если есть дробная часть"""
    if isinstance(value, (int, float)):
        return round(value, 2) if not value.is_integer() else int(value)
    return value

In [ ]:
# Функция для создания сводной таблицы результатов
def create_summary_table(results, instrument):
    """Создание сводной таблицы результатов"""
    metrics = ['final_equity', 'n_trades', 'returns', 'max_drawdown',
               'win_rate', 'sharpe_ratio', 'sortino_ratio', 'profit_factor']

    # Создаем MultiIndex для более структурированного отображения
    index_tuples = []
    data = []

    # Buy & Hold результаты
    if 'Buy & Hold' in results and 'full' in results['Buy & Hold']:
        bh_metrics = results['Buy & Hold']['full']
        index_tuples.append(('Buy & Hold', 'Full'))
        data.append({
            'Days': len(results['Buy & Hold']['full_data']),
            'Hyperparameters': 'N/A',
            **{metric: format_metric(bh_metrics[metric]) for metric in metrics}
        })

    # Результаты торговых стратегий
    #for strategy in ['Strategy2', 'Strategy3', 'Strategy1', 'Strategy4']:
    for strategy in ['Strategy4']:
        if strategy in results:
            for dataset in ['train_default', 'test','test_ml', 'test_neu', 'test_chronos', 'valid']:
                if dataset in results[strategy]:
                    metrics_data = results[strategy][dataset]

                    # Определяем, какие гиперпараметры использовать
                    if dataset == 'train_default':
                        hyperparams = results[strategy].get('train_default_params', 'N/A')
                    elif dataset == 'test':
                        hyperparams = results[strategy].get('optimized_params', 'N/A')
                    elif dataset == 'test_ml':
                        hyperparams = results[strategy].get('optimized_params_ml', 'N/A')
                    elif dataset == 'test_neu':
                        hyperparams = results[strategy].get('optimized_params_neu', 'N/A')
                    elif dataset == 'test_chronos':
                        hyperparams = results[strategy].get('optimized_params_chronos', 'N/A')
                    elif dataset == 'valid':
                        # Для Valid используем гиперпараметры, оптимизированные на объединенном наборе Train и Test
                        hyperparams = results[strategy].get('optimized_params_combined', 'N/A')

                    days = len(results[strategy][f'{dataset}_data'])

                    # Заменяем "train_default" на "Train"
                    dataset_name = 'Train' if dataset == 'train_default' else dataset.capitalize()

                    # Форматируем гиперпараметры с двумя знаками после точки, если есть дробная часть
                    formatted_hyperparams = {}
                    if hyperparams != 'N/A':
                        for param, value in hyperparams.items():
                            formatted_hyperparams[param] = round(value, 2) if isinstance(value, float) else value
                    else:
                        formatted_hyperparams = 'N/A'

                    index_tuples.append((strategy, dataset_name))
                    data.append({
                        'Days': days,
                        'Hyperparameters': formatted_hyperparams,
                        **{metric: format_metric(metrics_data[metric]) for metric in metrics}
                    })

    # Создаем DataFrame с MultiIndex
    df = pd.DataFrame(data, index=pd.MultiIndex.from_tuples(index_tuples, names=['Strategy', 'Dataset']))

    # Форматируем таблицу для красивого отображения
    styled_df = df.style\
        .format({
            'final_equity': '{:.2f}',
            'returns': '{:.2f}%',
            'max_drawdown': '{:.2f}%',
            'win_rate': '{:.2f}%',
            'sharpe_ratio': '{:.2f}',
            'sortino_ratio': '{:.2f}',
            'profit_factor': '{:.2f}'
        })\
        .set_caption(f'Сводная таблица результатов бэктестирования стратегий для {instrument}')\
        .set_table_styles([
            {'selector': 'caption', 'props': [('text-align', 'center'), ('font-size', '16px'), ('font-weight', 'bold')]},
            {'selector': 'th', 'props': [('text-align', 'center'), ('background-color', '#f0f0f0')]},
            {'selector': 'td', 'props': [('text-align', 'right')]}
        ])\
        .background_gradient(subset=['profit_factor'], cmap='RdYlGn')\
        .background_gradient(subset=['returns'], cmap='RdYlGn')

    return styled_df

In [ ]:
def analyze_trades(strategy, data, signals):
    """ Анализ фактических сделок на основе сигналов и ценовых данных. Возвращает список сделок с точками входа/выхода и прибылью. """
    trades = []
    in_position = False
    entry_price = 0
    entry_date = None
    position_type = None  # 'long' or 'short'
    for i in range(len(data)):
        current_date = data.index[i]
        current_price = data['Close'].iloc[i]
        is_dataframe = isinstance(signals, pd.DataFrame)
        if is_dataframe:
          current_signal = signals['Signal'].iloc[i]
        elif not is_dataframe:
          current_signal = signals.iloc[i]
        if not in_position and current_signal != 0:
            # Opening a position
            in_position = True
            entry_price = current_price
            entry_date = current_date
            position_type = 'long' if current_signal == 1 else 'short'
        elif in_position:
            # Check for closing conditions
            should_close = (
                (position_type == 'long' and current_signal == -1) or
                (position_type == 'short' and current_signal == 1) or
                (i == len(data) - 1)  # Close at the end of the period
            )
            if should_close:
                # Calculate profit
                profit = (
                    (current_price - entry_price) / entry_price * 100
                    if position_type == 'long'
                    else (entry_price - current_price) / entry_price * 100
                )
                trades.append({
                    'entry_date': entry_date,
                    'entry_price': entry_price,
                    'exit_date': current_date,
                    'exit_price': current_price,
                    'type': position_type,
                    'profit': profit
                })
                in_position = False
                entry_price = 0
                entry_date = None
                position_type = None
    return trades

In [ ]:
# Функция оценки стратегических показателей и расчета доходности с поправкой на риск
def calculate_risk_adjusted_returns(metrics):
  #https://www.wallstreetprep.com/knowledge/risk-adjusted-return/
    """Risk-Adjusted Returns позволяет выбрать стратегию, которая приносит доходность с минимальным риском..
       Он позволяет сравнивать стратегии не только по их абсолютной доходности, но и по тому, насколько эффективно они используют риск для достижения этой доходности.
       Оценить, насколько стратегия эффективно использует риск для достижения доходности. Это позволяет сравнивать стратегии, которые могут иметь схожую доходность, но разный уровень риска.
       Компоненты формулы: returns: Общая доходность стратегии. Чем выше доходность, тем выше значение метрики.
                           sharpe_ratio: Коэффициент Шарпа, который измеряет доходность с поправкой на общий риск (волатильность). Чем выше коэффициент Шарпа, тем лучше стратегия использует риск для получения доходности.
                           sortino_ratio: Коэффициент Сортино, который измеряет доходность с поправкой на риск убытков (негативную волатильность). Чем выше коэффициент Сортино, тем лучше стратегия управляет риском убытков.
                           profit_factor: Отношение общей прибыли к общему убытку. Чем выше значение, тем больше прибыль по сравнению с убытками.
                           max_drawdown: Максимальная просадка капитала. Чем меньше просадка, тем стабильнее стратегия. В формуле используется 1 + max_drawdown, чтобы избежать деления на ноль и уменьшить влияние больших просадок.
        Логика расчета: В числителе стоит произведение доходности, коэффициента Шарпа, коэффициента Сортино и фактора прибыли. Это позволяет учитывать как абсолютную доходность, так и эффективность управления риском.
                        В знаменателе - метрика максимальной просадки, что снижает значение метрики, если стратегия имеет высокий уровень риска (большие просадки).
    """
    if not metrics:
        return float('-inf')

    # Нормализуем доходность по показателям риска
    risk_adjusted_return = (
        metrics['returns'] *
        metrics['sharpe_ratio'] *
        metrics['sortino_ratio']
        #metrics['profit_factor']
    ) / (1 + metrics['max_drawdown'])  # Чтобы избежать деления на ноль.

    return risk_adjusted_return

In [ ]:

def plot_best_strategy(data, strategy, signals, equity_curve, strategy_results, trades, title):
    """ Функция визуализации, показывающая реальные сделки стратегии вместо сигналов """
    fig = make_subplots(rows=2, cols=1,
                        shared_xaxes=True,
                        vertical_spacing=0.03,
                        row_heights=[0.7, 0.3],
                        subplot_titles=(
                            "Price and Trades",  # Убираем дублирование заголовка
                            "Equity Curve"
                        ))

    # Candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name='Price'
        ),
        row=1, col=1
    )

    # Отображение сделок
    long_entries = []
    long_exits = []
    short_entries = []
    short_exits = []

    for trade in trades:
        if trade['type'] == 'long':
            long_entries.append({
                'date': trade['entry_date'],
                'price': trade['entry_price'],
                'profit': trade['profit']
            })
            long_exits.append({
                'date': trade['exit_date'],
                'price': trade['exit_price'],
                'profit': trade['profit']
            })
        else:  # short trade
            short_entries.append({
                'date': trade['entry_date'],
                'price': trade['entry_price'],
                'profit': trade['profit']
            })
            short_exits.append({
                'date': trade['exit_date'],
                'price': trade['exit_price'],
                'profit': trade['profit']
            })

    # Отображение сделок long
    if long_entries:
        fig.add_trace(
            go.Scatter(
                x=[t['date'] for t in long_entries],
                y=[t['price'] * 0.99 for t in long_entries],
                mode='markers',
                marker=dict(
                    symbol='triangle-up',
                    size=12,
                    color='green',
                    line=dict(width=2)
                ),
                name='Long Entry',
                text=[f"Long Entry<br>Date: {t['date'].strftime('%Y-%m-%d')}<br>Price: {t['price']:.2f}<br>Profit: {t['profit']:.2f}%"
                      for t in long_entries],
                hoverinfo='text'
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=[t['date'] for t in long_exits],
                y=[t['price'] * 1.01 for t in long_exits],
                mode='markers',
                marker=dict(
                    symbol='triangle-down',
                    size=12,
                    color='blue',
                    line=dict(width=2)
                ),
                name='Long Exit',
                text=[f"Long Exit<br>Date: {t['date'].strftime('%Y-%m-%d')}<br>Price: {t['price']:.2f}<br>Profit: {t['profit']:.2f}%"
                      for t in long_exits],
                hoverinfo='text'
            ),
            row=1, col=1
        )

    # Отображение сделок Short
    if short_entries:
        fig.add_trace(
            go.Scatter(
                x=[t['date'] for t in short_entries],
                y=[t['price'] * 1.01 for t in short_entries],
                mode='markers',
                marker=dict(
                    symbol='triangle-down',
                    size=12,
                    color='red',
                    line=dict(width=2)
                ),
                name='Short Entry',
                text=[f"Short Entry<br>Date: {t['date'].strftime('%Y-%m-%d')}<br>Price: {t['price']:.2f}<br>Profit: {t['profit']:.2f}%"
                      for t in short_entries],
                hoverinfo='text'
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=[t['date'] for t in short_exits],
                y=[t['price'] * 0.99 for t in short_exits],
                mode='markers',
                marker=dict(
                    symbol='triangle-up',
                    size=12,
                    color='orange',
                    line=dict(width=2)
                ),
                name='Short Exit',
                text=[f"Short Exit<br>Date: {t['date'].strftime('%Y-%m-%d')}<br>Price: {t['price']:.2f}<br>Profit: {t['profit']:.2f}%"
                      for t in short_exits],
                hoverinfo='text'
            ),
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          row=1, col=1
        )

    # Equity curve
    if isinstance(equity_curve, list):
        equity_curve = np.array(equity_curve)

    # Проверка соответствия equity_curve длине набора данных
    if len(equity_curve) > len(data):
        equity_curve = equity_curve[-len(data):]
    elif len(equity_curve) < len(data):
        padding = np.full(len(data) - len(equity_curve), equity_curve[-1])
        equity_curve = np.concatenate([equity_curve, padding])

    fig.add_trace(
        go.Scatter(
            x=data.index,
            y=equity_curve,
            name='Equity',
            line=dict(color='blue', width=2),
            text=[f"Equity: {eq:.2f}" for eq in equity_curve],
            hoverinfo='text+x'
        ),
        row=2, col=1
    )

    # Настройки графика
    fig.update_layout(
        title=dict(
            text=title,  # Заголовок передается только здесь
            x=0.5,
            xanchor='center'
        ),
        showlegend=True,
        height=800,
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )

    # Настройки осей графика
    fig.update_xaxes(
        title_text=" ",
        rangeslider_visible=False,
        gridcolor='lightgrey',
        showgrid=True
    )

    fig.update_yaxes(
        title_text="Price",
        gridcolor='lightgrey',
        showgrid=True,
        row=1, col=1
    )

    fig.update_yaxes(
        title_text="Equity",
        gridcolor='lightgrey',
        showgrid=True,
        row=2, col=1
    )

    return fig

In [ ]:
# расчет и визуализация roc auc
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle
import numpy as np
def is_float_in_range(lower_bound, value, upper_bound):
    return lower_bound <= value <= upper_bound

###!!!! не работает
def rocauc_torch(X_test, y_test):
  auc_score = roc_auc_score(y_test, y_test)
  print(f"ROC AUC Score: {auc_score:.4f}")
  fpr, tpr, thresholds = roc_curve(y_test, y_test)

  if is_float_in_range(0.9, auc_score, 1.0) :
    print(f"Модель высокой надежности, пригодная для критически важных задач")
  elif is_float_in_range(0.8, auc_score, 0.9):
    print(f"Подходит для большинства бизнес-задач")
  elif is_float_in_range(0.7, auc_score, 0.8):
    print(f"Пригодна для некритичных задач; требует дополнительной проверки")
  elif is_float_in_range(0.6, auc_score, 0.7):
    print(f"Требует значительного улучшения")
  elif is_float_in_range(0.5, auc_score, 0.6):
    print(f"Немного лучше случайного угадывания")
  elif is_float_in_range(0.0, auc_score, 0.5):
    print(f"Эквивалентна случайному угадыванию")

  return auc_score

def rocauc(model, X_test, y_test):
  y_pred_proba = model.predict_proba(X_test)[:, 1]
  auc_score = roc_auc_score(y_test, y_pred_proba)
  print(f"ROC AUC Score: {auc_score:.4f}")
  fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

  if is_float_in_range(0.9, auc_score, 1.0) :
    print(f"Модель высокой надежности, пригодная для критически важных задач")
  elif is_float_in_range(0.8, auc_score, 0.9):
    print(f"Подходит для большинства бизнес-задач")
  elif is_float_in_range(0.7, auc_score, 0.8):
    print(f"Пригодна для некритичных задач; требует дополнительной проверки")
  elif is_float_in_range(0.6, auc_score, 0.7):
    print(f"Требует значительного улучшения")
  elif is_float_in_range(0.5, auc_score, 0.6):
    print(f"Немного лучше случайного угадывания")
  elif is_float_in_range(0.0, auc_score, 0.5):
    print(f"Эквивалентна случайному угадыванию")

  return auc_score
  #plt.figure(figsize=(8, 6))
  #plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc_score:.2f})')
  #plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
  #plt.xlim([0.0, 1.0])
  #plt.ylim([0.0, 1.05])
  #plt.xlabel('False Positive Rate')
  #plt.ylabel('True Positive Rate')
  #plt.title('Receiver Operating Characteristic (ROC) Curve')
  #plt.legend(loc="lower right")
  #plt.grid(True)
  #plt.show()


In [ ]:
# Функция визуализации результатов лучшей стратегии с сигналами
def plot_best_strategy_signals(data, strategy, signals, strategy_results, title1):
    """Функция визуализации результатов лучшей стратегии с отдельными подграфиками для каждого индикатора"""
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.droplevel(1)

    # Определяем количество подграфиков в зависимости от стратегии
    if isinstance(strategy, Strategy1):
        # Для Strategy1: EMA, RSI, MACD
        num_subplots = 4  # Основной график + 3 индикатора
        subplot_titles = ["Price и сигналы", "EMA", "RSI", "MACD"]  # Убираем дублирование заголовка
    elif isinstance(strategy, Strategy2):
        # Для Strategy2: Bollinger Bands, Stochastic + RSI
        num_subplots = 4  # Основной график + 3 индикатора
        subplot_titles = ["Price и сигналы", "Bollinger Bands", "Stochastic %K", "Stochastic %D", "RSI"]  # Убираем дублирование заголовка
    elif isinstance(strategy, Strategy3):
        # Для Strategy3: RSI
        num_subplots = 2  # Основной график + 1 индикатор
        subplot_titles = ["Price и сигналы", "RSI"]  # Убираем дублирование заголовка
    elif isinstance(strategy, Strategy4):
        # Для Strategy4: RSI
        num_subplots = 5  # Основной график + 5 индикаторов
        subplot_titles = ["Price и сигналы", "RSI", "EMA", "MACD", "Bollinger Bands"]  # Убираем дублирование заголовка
    else:
        num_subplots = 2  # По умолчанию: основной график + 1 индикатор
        subplot_titles = ["Price и сигналы", "Indicator"]  # Убираем дублирование заголовка

    # Создание subplot с несколькими графиками
    fig = make_subplots(rows=num_subplots, cols=1,
                        shared_xaxes=True,
                        vertical_spacing=0.05,
                        row_heights=[0.6] + [0.1] * (num_subplots - 1),  # Основной график занимает больше места
                        subplot_titles=subplot_titles)  # Заголовки для каждого подграфика

    # Добавление свечного графика на первый подграфик
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name='Цена'
        ),
        row=1, col=1
    )

    # Добавление сигналов на первый подграфик
    buy_signals = signals[signals == 1].index
    sell_signals = signals[signals == -1].index

    # Покупки
    if len(buy_signals) > 0:
        fig.add_trace(
            go.Scatter(
                x=buy_signals,
                y=data.loc[buy_signals, 'Low'] * 0.99,
                mode='markers',
                marker=dict(
                    symbol='triangle-up',
                    size=12,
                    color='green',
                    line=dict(width=2)
                ),
                name='Сигнал на покупку',
                text=[f"Покупка<br>Дата: {date.strftime('%Y-%m-%d')}<br>Цена: {data.loc[date, 'Close']:.2f}"
                      for date in buy_signals],
                hoverinfo='text'
            ),
            row=1, col=1
        )

    # Продажи
    if len(sell_signals) > 0:
        fig.add_trace(
            go.Scatter(
                x=sell_signals,
                y=data.loc[sell_signals, 'High'] * 1.01,
                mode='markers',
                marker=dict(
                    symbol='triangle-down',
                    size=12,
                    color='red',
                    line=dict(width=2)
                ),
                name='Сигнал на продажу',
                text=[f"Продажа<br>Дата: {date.strftime('%Y-%m-%d')}<br>Цена: {data.loc[date, 'Close']:.2f}"
                      for date in sell_signals],
                hoverinfo='text'
            ),
            row=1, col=1
        )

    # Добавление индикаторов на отдельные подграфики
    if isinstance(strategy, Strategy1):
        # Для Strategy1: EMA, RSI, MACD
        close_prices = data['Close'].values.astype(float).flatten()
        df = data.copy()
        df['EMA'] = ta.EMA(close_prices, timeperiod=strategy.ema_period)
        df['RSI'] = ta.RSI(close_prices, timeperiod=strategy.rsi_period)
        macd, signal, _ = ta.MACD(close_prices,
                                  fastperiod=strategy.macd_fast,
                                  slowperiod=strategy.macd_slow,
                                  signalperiod=strategy.macd_signal)
        df['MACD'] = macd
        df['Signal'] = signal

        # EMA на втором подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['EMA'],
                name='EMA',
                line=dict(color='blue', width=1)
            ),
            row=2, col=1
        )

        # RSI на третьем подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['RSI'],
                name='RSI',
                line=dict(color='orange', width=1)
            ),
            row=3, col=1
        )

        # MACD и Signal на четвертом подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['MACD'],
                name='MACD',
                line=dict(color='green', width=1)
            ),
            row=4, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['Signal'],
                name='MACD Signal',
                line=dict(color='red', width=1)
            ),
            row=4, col=1
        )

    elif isinstance(strategy, Strategy2):
        # Для Strategy2: Bollinger Bands, Stochastic + RSI
        close_prices = data['Close'].values.astype(float).flatten()
        high_prices = data['High'].values.astype(float).flatten()
        low_prices = data['Low'].values.astype(float).flatten()
        df = data.copy()
        df['BB_middle'] = ta.SMA(close_prices, timeperiod=strategy.bb_period)
        std_dev = ta.STDDEV(close_prices, timeperiod=strategy.bb_period)
        df['RSI'] = ta.RSI(close_prices, timeperiod=strategy.rsi_period)
        df['BB_upper'] = df['BB_middle'] + strategy.bb_std * std_dev
        df['BB_lower'] = df['BB_middle'] - strategy.bb_std * std_dev
        slowk, slowd = ta.STOCH(high_prices,
                               low_prices,
                               close_prices,
                               fastk_period=strategy.stoch_k,
                               slowk_period=strategy.stoch_d,
                               slowd_period=strategy.stoch_slow)
        df['slowk'] = slowk
        df['slowd'] = slowd

        # Bollinger Bands на втором подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['BB_upper'],
                name='BB Upper',
                line=dict(color='blue', width=1)
            ),
            row=2, col=1
        )

        # RSI на третьем подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['RSI'],
                name='RSI',
                line=dict(color='orange', width=1)
            ),
            row=3, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['BB_middle'],
                name='BB Middle',
                line=dict(color='green', width=1)
            ),
            row=2, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['BB_lower'],
                name='BB Lower',
                line=dict(color='red', width=1)
            ),
            row=2, col=1
        )

        # Stochastic на третьем подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['slowk'],
                name='Stochastic %K',
                line=dict(color='purple', width=1)
            ),
            row=3, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['slowd'],
                name='Stochastic %D',
                line=dict(color='orange', width=1)
            ),
            row=3, col=1
        )


    elif isinstance(strategy, Strategy3):
        # Для Strategy3: RSI
        close_prices = data['Close'].values.astype(float).flatten()
        df = data.copy()
        df['RSI'] = ta.RSI(close_prices, timeperiod=strategy.rsi_period)

        # RSI на втором подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['RSI'],
                name='RSI',
                line=dict(color='blue', width=1)
            ),
            row=2, col=1
        )
    elif isinstance(strategy, Strategy4):
        # Для Strategy4
        close_prices = data['Close'].values.astype(float).flatten()
        df = data.copy()
        df['RSI'] = ta.RSI(close_prices, timeperiod=strategy.rsi_period)
        df['BB_middle'] = ta.SMA(close_prices, timeperiod=strategy.bb_period)
        std_dev = ta.STDDEV(close_prices, timeperiod=strategy.bb_period)
        df['BB_upper'] = df['BB_middle'] + strategy.bb_std * std_dev
        df['BB_lower'] = df['BB_middle'] - strategy.bb_std * std_dev
        df['EMA'] = ta.EMA(close_prices, timeperiod=strategy.ema_period)
        macd, signal, _ = ta.MACD(close_prices,
                                  fastperiod=strategy.macd_fast,
                                  slowperiod=strategy.macd_slow,
                                  signalperiod=strategy.macd_signal)
        df['MACD'] = macd
        df['Signal'] = signal


        # RSI на втором подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['RSI'],
                name='RSI',
                line=dict(color='blue', width=1)
            ),
            row=2, col=1
        )

        # Bollinger Bands на пятом подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['BB_upper'],
                name='BB Upper',
                line=dict(color='blue', width=1)
            ),
            row=5, col=1
        )

        # EMA на втором подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['EMA'],
                name='EMA',
                line=dict(color='blue', width=1)
            ),
            row=3, col=1
        )

        # MACD и Signal на четвертом подграфике
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['MACD'],
                name='MACD',
                line=dict(color='green', width=1)
            ),
            row=4, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['Signal'],
                name='MACD Signal',
                line=dict(color='red', width=1)
            ),
            row=4, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['BB_middle'],
                name='BB Middle',
                line=dict(color='green', width=1)
            ),
            row=5, col=1
        )

        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=df['BB_lower'],
                name='BB Lower',
                line=dict(color='red', width=1)
            ),
            row=5, col=1
        )

    # Настройка макета
    fig.update_layout(
        title=dict(
            text=title1,  # Заголовок передается только здесь
            x=0.5,
            xanchor='center'
        ),
        showlegend=True,
        height=200 * num_subplots,  # Высота графика зависит от количества подграфиков
        template='plotly_white',
        hovermode='x unified',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01
        )
    )

    # Настройка осей
    fig.update_xaxes(
        title_text=" ",
        rangeslider_visible=False,
        gridcolor='lightgrey',
        showgrid=True
    )

    fig.update_yaxes(
        title_text=" ",
        gridcolor='lightgrey',
        showgrid=True,
        row=1, col=1
    )

    # Настройка осей для индикаторов
    for i in range(2, num_subplots + 1):
        fig.update_yaxes(
            title_text=" ",
            gridcolor='lightgrey',
            showgrid=True,
            row=i, col=1
        )

    return fig

In [ ]:
# Функция для бэктестирования стратегии
def backtest_strategy(strategy, data):
    """Бэктестирование стратегии"""
    try:
        if data is None or len(data) < 2:
            return None

        signals, conditions, df = strategy.generate_signals(data)
        if signals is None or signals.empty:
            return None


        signals = signals.fillna(0)
        signals = signals.reindex(data.index, fill_value=0)

        close_prices = data['Close'].values
        equity = 1.0
        equity_curve = [equity]
        positions = []
        position_open = False
        entry_price = 0
        entry_signal = 0
        entry_date = None

        for i in range(1, len(signals)):
            current_signal = signals.iloc[i]
            current_date = signals.index[i]

            if not position_open and current_signal != 0:
                # Открываем позицию
                position_open = True
                entry_price = close_prices[i]
                entry_signal = current_signal
                entry_date = current_date

            elif position_open and ((current_signal != 0 and current_signal != entry_signal) or i == len(signals)-1):
                # Закрываем позицию
                exit_price = close_prices[i]
                profit = (exit_price - entry_price) / entry_price * entry_signal

                positions.append({
                    'entry_date': entry_date,
                    'exit_date': current_date,
                    'profit': float(profit)
                })

                equity *= (1.0 + profit)
                position_open = False

            equity_curve.append(float(equity))

        strategy.positions = positions
        strategy.equity = equity_curve
        strategy.signals = signals  # Сохраняем сигналы для последующей визуализации

        return strategy.calculate_metrics(data), conditions, signals, df

    except Exception as e:
        print(f"Ошибка при бэктестировании: {str(e)}")
        return None

In [ ]:
def find_best_strategy(all_results):
    """ Поиск лучшей стратегии среди всех инструментов и наборов данных на основе доходности с поправкой на риск. """
    best_strategy = {
        'strategy_name': None,
        'instrument': None,
        'dataset': None,
        'metrics': None,
        'risk_adjusted_returns': float('-inf'),
        'params': None
    }

    for instrument, results in all_results.items():
        #for strategy_name in ['Strategy1', 'Strategy2', 'Strategy3', 'Strategy4']:
        for strategy_name in ['Strategy4']:
            if strategy_name not in results:
                continue

            # Check all datasets (train_default, test, valid)
            for dataset in ['train_default', 'test', 'test_ml','test_neu','test_chronos', 'valid']:
                if dataset not in results[strategy_name]:
                    continue

                metrics = results[strategy_name][dataset]
                risk_adjusted_returns = calculate_risk_adjusted_returns(metrics)

                if risk_adjusted_returns > best_strategy['risk_adjusted_returns']:
                    params_key = 'train_default_params' if dataset == 'train_default' else 'optimized_params'
                    if dataset == 'valid':
                        params_key = 'optimized_params_combined'

                    best_strategy = {
                        'strategy_name': strategy_name,
                        'instrument': instrument,
                        'dataset': dataset,
                        'metrics': metrics,
                        'risk_adjusted_returns': risk_adjusted_returns,
                        'params': results[strategy_name].get(params_key)
                    }

    return best_strategy

In [ ]:
def analyze_instrument(symbol):
    """Анализ торговых стратегий для одного инструмента"""

    try:
        data = load_data(symbol)
        if data is None or len(data) < 100:
            print(f"Недостаточно данных для {symbol}")
            return None, None

        data = data.copy()
        data = data.ffill().copy()
        #data = data.fillna(method='ffill').fillna(method='bfill')

        if data.isnull().any().any():
            print(f"Обнаружены пропущенные значения после обработки в {symbol}")
            return None, None

        print(f"\nРазмер данных: {len(data)} строк")

        train_data, test_data, valid_data = split_data(data)

        print("\nПериоды наборов данных:")
        print(f"Train: {train_data.index[0].date()} - {train_data.index[-1].date()}")
        print(f"Test: {test_data.index[0].date()} - {test_data.index[-1].date()}")
        print(f"Valid: {valid_data.index[0].date()} - {valid_data.index[-1].date()}")
        print(f"TestMl: {test_data.index[0].date()} - {test_data.index[-1].date()}")


        results = {}

        try:
            bh_results = backtest_buy_hold(data)
            if bh_results:
                results['Buy & Hold'] = {
                    'full': bh_results,
                    'full_data': data  # Сохраняем данные для расчета количества дней
                }
        except Exception as e:
            print(f"Ошибка при расчете Buy & Hold для {symbol}: {str(e)}")

        for strategy_name, strategy_params in [
            #('Strategy1', Strategy1()),
            #('Strategy2', Strategy2()),
            #('Strategy3', Strategy3()),
            ('Strategy4', Strategy4())
        ]:
            try:
                strategy_results = {}

                # Бэктестирование на Train с гиперпараметрами по умолчанию
                print(f"\n1. Набор данных Train. Начало бэктестирования {strategy_name} с гиперпараметрами по умолчанию: {strategy_params.get_params()}")
                #print(f"Гиперпараметры по умолчанию: {strategy_params.get_params()}")
                train_metrics, conditions, signals, df = backtest_strategy(strategy_params, train_data)
                if train_metrics:
                    strategy_results['train_default'] = train_metrics
                    strategy_results['train_default_params'] = strategy_params.get_params()  # Гиперпараметры
                    strategy_results['train_default_data'] = train_data  # Данные для расчета количества дней

                # Оптимизация гиперпараметров на Train
                print(f"\nНачало оптимизации гиперпараметров для {strategy_name} на Train...")
                param_space = param_spaces[strategy_name]
                best_params = optimize_strategy(globals()[strategy_name], train_data, param_space)
                if best_params:
                    print(f"Оптимальные гиперпараметры для {strategy_name} на Train: {best_params}")
                    optimized_strategy = globals()[strategy_name](**best_params)
                    optimized_strategy.ticker = symbol
                    strategy_results['optimized_params'] = best_params  # Оптимизированные гиперпараметры

                    # Бэктестирование на Test с оптимизированными гиперпараметрами
                    print(f"\n2. Набор данных Test. Начало бэктестирования {strategy_name} с оптимизированными гиперпараметрами на наборе Train: {best_params}")
                    #print(f"Оптимизированные гиперпараметры: {best_params}")
                    test_metrics, conditions_test, signals_test, df_test = backtest_strategy(optimized_strategy, test_data)
                    if test_metrics:
                        strategy_results['test'] = test_metrics
                        strategy_results['test_data'] = test_data  # Данные для расчета количества дней

                    #!!!!! Объединение Train и Test для оптимизации
                    #!!!!!!!!!!4. Ячейка 38: split_data формально ок, но дальше ломается логика эксперимента
                    #train_test_split(..., shuffle=False) - это нормально как “простое временное разбиение”.
                    #Но проблема в том, как test используется далее:
                    #В analyze_instrument (ячейка 38) ты делаешь:
                    #- оптимизация на Train -> проверка на Test - это ОК
                    #- затем объединяешь Train+Test, оптимизируешь снова (best_params_combined)
                    #- и после этого снова прогоняешь вещи на Test (и даже ML/NN на test) - это уже нечестно, потому что test стал частью тюнинга модели.

                    train_test_data = pd.concat([train_data, test_data])
                    print(f"\nОбъединенный набор данных Train и Test: {train_test_data.index[0].date()} - {train_test_data.index[-1].date()}")

                    # Оптимизация гиперпараметров на объединенном наборе
                    print(f"\nНачало оптимизации гиперпараметров для {strategy_name} на объединенном наборе Train и Test...")
                    best_params_combined = optimize_strategy(globals()[strategy_name], train_test_data, param_space)
                    print(f"Оптимальные гиперпараметры для {strategy_name} на объединенном наборе Train и Test: {best_params_combined}")
                    optimized_strategy_combined = globals()[strategy_name](**best_params_combined)
                    strategy_results['optimized_params_combined'] = best_params_combined  # Оптимизированные гиперпараметры

                    # проверка стратегии с испльзованием ML
                    print(f"\n3. Набор данных Test.  Начало бэктестирования {strategy_name} с оптимизированными гиперпараметрами на наборе Test: {best_params_combined} с использованием ML")
                    ml_metrics = backtestWithML(optimized_strategy, data, test_data)
                    #display(ml_metrics)
                    if ml_metrics:
                        strategy_results['test_ml'] = ml_metrics.copy()
                        strategy_results['test_ml_data'] = test_data.copy()  # Данные для расчета количества дней

                    # проверка стратегии с использованием нейронной сети txmixer
                    print(f"\n4. Набор данных Test. Модель TSMixer. Начало бэктестирования {strategy_name} с оптимизированными гиперпараметрами на наборе Test: {best_params_combined} с использованием ML")
                    neu_metrics = backtestWithNeu(optimized_strategy, data, test_data)
                    if neu_metrics:
                        strategy_results['test_neu'] = neu_metrics.copy()
                        strategy_results['test_neu_data'] = test_data.copy()  # Данные для расчета количества дней

                    # проверка стратегии с испльзованием нейронной сети chronos
                    print(f"\n5. Набор данных Test. Модель Chronos. Начало бэктестирования {strategy_name} с оптимизированными гиперпараметрами на наборе Test: {best_params_combined} с использованием ML")
                    chronos_metrics = backtestWithChronos(optimized_strategy, data, test_data, action="Load")
                    if chronos_metrics:
                        strategy_results['test_chronos'] = chronos_metrics.copy()
                        strategy_results['test_chronos_data'] = test_data.copy()  # Данные для расчета количества дней

                    # Бэктестирование на Valid с оптимизированными гиперпараметрами
                    print(f"\n6. Набор данных Valid. Начало бэктестирования {strategy_name} с оптимизированными гиперпараметрами на объединенном наборе Train и Test: {best_params_combined}")
                    #print(f"Оптимизированные гиперпараметры: {best_params_combined}")
                    #valid_metrics, conditions, signals, df = backtest_strategy(optimized_strategy_combined, valid_data)
                    valid_metrics = backtestWithNeu(optimized_strategy, data, valid_data, action="Load")
                    if valid_metrics:
                        strategy_results['valid'] = valid_metrics
                        strategy_results['valid_data'] = valid_data  # Данные для расчета количества дней
                    print("=" * 100)  # Сплошная горизонтальная линия

                if strategy_results:
                    results[strategy_name] = strategy_results

            except Exception as e:
                print(f"Ошибка при анализе стратегии {strategy_name}: {str(e)}")
                continue

        if not results:
            print(f"Не удалось получить результаты для {symbol}")
            return None, None


        return results, data

    except Exception as e:
        print(f"Ошибка при анализе инструмента {symbol}: {str(e)}")
        return None, None


# Функция отображения графика датафрейма
def display_graf(df):
  sns.set(style="darkgrid")
  plt.figure(figsize=(15, 6))
  #display(df)
  dfgr = df
  #dfgr['short_mavg'] = dfgr['Close'].rolling(50).mean()
  #dfgr['long_mavg'] = dfgr['Close'].rolling(200).mean()
  subset = dfgr[dfgr['Ticker'] == 'GOOG']
  plot_data = subset[-500:]
  plt.plot(plot_data.index, plot_data['Close'], label='GOOG')
  plt.title(' ')
  plt.xlabel('Date')
  plt.ylabel('Close Price')
  plt.legend()
  plt.show()


# Функция загрузки данных
def load_data(symbol, start_date='2022-01-01', end_date='2026-01-01'):
    """Загрузка данных с Yahoo Finance"""
    display(f"\nЗагрузка данных с Yahoo Finance {symbol}")

    df = yf.download(symbol, start=start_date, end=end_date, interval='1d', auto_adjust=True, multi_level_index=False, ignore_tz=True)#, interval='1h') progress=False, multi_level_index=False)

    df.index.name = 'Date'
    #df.index.tz_convert(None)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    #pd.to_datetime(df.index, format='%Y-%m-%d %H:%M:%S')
    #df=df[df.Volume!=0]

    #df.index = pd.to_datetime(df.index, format='%Y-%m-%d %H:%M:%S')
    df.index = pd.to_datetime(df.index, format = '%Y-%m-%d')
    df = df.dropna()  # Удаление строк с пропущенными значениями
    df = df[~df.index.duplicated()]  # Удаление дубликатов


    # Перевод всех значений в числовой формат
    for c in [x for x in df.columns if x != 'Date']:
        df[c] = df[c].apply(pd.to_numeric, errors='coerce')
    # Сбрасываем мультииндекс, если он есть
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    return df


# Функция разделения данных на наборы
def split_data(df):
    """Разделение данных на наборы данных: Train, Test и Valid"""
    train_size = 0.6
    test_size = 0.2
    # Первое разделение на train и остаток
    train_data, temp_data = train_test_split(df, train_size=train_size, shuffle=False)
    #!!!!!!!!!4. Ячейка 38: split_data формально ок, но дальше ломается логика эксперимента
    #train_test_split(..., shuffle=False) - это нормально как “простое временное разбиение”.
    #Но проблема в том, как test используется далее:
    #В analyze_instrument (ячейка 38) ты делаешь:
    #- оптимизация на Train -> проверка на Test - это ОК
    #- затем объединяешь Train+Test, оптимизируешь снова (best_params_combined)
    #- и после этого снова прогоняешь вещи на Test (и даже ML/NN на test) - это уже нечестно, потому что test стал частью тюнинга модели.
    # Разделение остатка на test и valid
    test_size_adjusted = test_size / (1 - train_size)
    test_data, valid_data = train_test_split(temp_data, train_size=test_size_adjusted, shuffle=False)
    #train_data = train_data.reindex(pd.date_range(start=train_data.index.min(),end=train_data.index.max(),freq='1D'))
    #train_data = train_data.interpolate(method='linear').ffill().copy()

    return train_data, test_data, valid_data



def main():
    """Основная функция с выбором стратегии и визуализацией"""
    all_results = {}
    # Определение списка инструментов
    stocks = ['AAPL', 'MSFT', 'GOOGL', 'NVDA', 'BTC-USD']
    #cryptos = ['SOL-USD', 'BTC-USD']

    instruments = stocks
    for instrument in instruments:

        print(f"\n\033[1;34mАнализ инструмента: {instrument}\033[0m")
        print("=" * 100)  # Сплошная горизонтальная линия
        try:
            results, data = analyze_instrument(instrument)
            if results is not None and data is not None:
                all_results[instrument] = results
                summary_table = create_summary_table(results, instrument)
                display(summary_table)

        except Exception as e:
            print(f"Ошибка в анализе инструмента {instrument}: {str(e)}")
            continue

    # Поиск лучшей стратегии, используя метод отбора
    best_strategy = find_best_strategy(all_results)

    if best_strategy['strategy_name']:
        print(f"\n\033[1;34mОпределение стратегии с лучшими показателями инструменту и по набору данных\033[0m")
        print(f"Лучшая стратегия: {best_strategy['strategy_name']}")
        print(f"Лучший инструмент: {best_strategy['instrument']}")
        print(f"Лучший набор данных: {best_strategy['dataset']}")
        print(f"Доходность с корректировкой риска (Risk-Adjusted Returns): {best_strategy['risk_adjusted_returns']:.2f}")
        print("\nОптимальные гиперпараметры:")
        for param, value in best_strategy['params'].items():
            print(f"{param}: {value}")

        # Вывод метрик производительности
        metrics = best_strategy['metrics']
        print("\nМетрики производительности для лучшей стратегии:")
        print(f"Конечный капитал (final_equity) : {metrics['final_equity']:.2f}")
        print(f"Количество сделок (n_trades): {metrics['n_trades']}")
        print(f"Доходность (returns): {metrics['returns']:.2f}%")
        print(f"Максимальная просадка (max_drawdown): {metrics['max_drawdown']:.2f}%")
        print(f"Процент прибыльных сделок (win_rate): {metrics['win_rate']:.2f}%")
        print(f"Коэффициент Шарпа (sharpe_ratio): {metrics['sharpe_ratio']:.2f}")
        print(f"Коэффициент Сортино (sortino_ratio): {metrics['sortino_ratio']:.2f}")
        print(f"Фактор прибыли (profit_factor): {metrics['profit_factor']:.2f}")

        # Создание объекта стратегии с оптимальными параметрами
        strategy_class = globals()[best_strategy['strategy_name']]
        best_strategy_obj = strategy_class(**best_strategy['params'])

        # Получение набора данных
        data = load_data(best_strategy['instrument'])
        train_data, test_data, valid_data = split_data(data)
        if best_strategy['dataset'] == 'train_default':
            best_data = train_data.copy()
            backtest_strategy(best_strategy_obj, best_data)
            trades = analyze_trades(best_strategy_obj, best_data, best_strategy_obj.signals)
        elif best_strategy['dataset'] == 'test':
            best_data = test_data.copy()
            backtest_strategy(best_strategy_obj, best_data)
            trades = analyze_trades(best_strategy_obj, best_data, best_strategy_obj.signals)
        elif best_strategy['dataset'] == 'test_ml':
            best_data = test_data.copy()
            test_metrics = backtestWithML(best_strategy_obj, train_data, best_data, ml_modelname=metrics['ml_modelname'])
            metrics = test_metrics
            trades = analyze_trades(best_strategy_obj, best_data, best_strategy_obj.signals)
            #backtest_strategy(best_strategy_obj, best_data)
        elif best_strategy['dataset'] == 'test_neu':
            best_data = test_data.copy()
            test_metrics = backtestWithNeu(best_strategy_obj, train_data, best_data, action="Load", instrument=best_strategy['instrument'])
            metrics = test_metrics
            #trades = analyze_trades(best_strategy_obj, best_data, best_strategy_obj.signals)
        elif best_strategy['dataset'] == 'test_chronos':
            best_data = test_data.copy()
            test_metrics = backtestWithChronos(best_strategy_obj, train_data, best_data, action="Load", instrument=best_strategy['instrument'])
            metrics = test_metrics
            #trades = analyze_trades(best_strategy_obj, best_data, best_strategy_obj.signals)
        else:
            best_data = valid_data.copy()
            valid_metrics = backtestWithNeu(best_strategy_obj, train_data, best_data, action="Load", instrument=best_strategy['instrument'])
            metrics = valid_metrics
            #trades = analyze_trades(best_strategy_obj, best_data, best_strategy_obj.signals)


    else:
        print("\nНе найдена лучшая стратегия")

if __name__ == "__main__":

  main()


Анализ инструмента: AAPL


'\nЗагрузка данных с Yahoo Finance AAPL'

[*********************100%***********************]  1 of 1 completed


Размер данных: 1003 строк

Периоды наборов данных:
Train: 2022-01-03 - 2024-05-23
Test: 2024-05-24 - 2025-03-14
Valid: 2025-03-17 - 2025-12-31
TestMl: 2024-05-24 - 2025-03-14

1. Набор данных Train. Начало бэктестирования Strategy4 с гиперпараметрами по умолчанию: {'bb_period': 20, 'bb_std': 2.0, 'rsi_period': 14, 'sma': 10, 'ema_period': 4, 'ma_period': 10, 'macd_fast': 12, 'macd_slow': 26, 'macd_signal': 9, 'stoch_k': 14, 'stoch_d': 3, 'stoch_slow': 3}

Начало оптимизации гиперпараметров для Strategy4 на Train...
  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

100%|██████████| 50/50 [00:05<00:00,  9.75trial/s, best loss: -1.4025422886110488]
Оптимальные гиперпараметры для Strategy4 на Train: {'bb_period': np.float64(32.0), 'bb_std': np.float64(2.743394926900637), 'ema_period': np.float64(25.0), 'ma_period': np.float64(35.0), 'macd_fast': np.float64(10.0), 'macd_signal': np.float64(15.0), 'macd_slow': np.float64(40.0), 'rsi_period': np.float64(11.0), 'sma': np.float64(50.0), 'stoch_d': np.float64(3.0), 'stoch_k': np.float64(13.0), 'stoch_slow': np.float64(5.0)}

2. Набор данных Test. Начало бэктестирования Strategy4 с оптимизированными гиперпараметрами на наборе Train: {'bb_period': np.float64(32.0), 'bb_std': np.float64(2.743394926900637), 'ema_period': np.float64(25.0), 'ma_period': np.float64(35.0), 'macd_fast': np.float64(10.0), 'macd_signal': np.float64(15.0), 'macd_slow': np.float64(40.0), 'rsi_period': np.float64(11.0), 'sma': np.float64(50.0), 'stoch_d': np.float64(3.0), 'stoch_k': np.float64(13.0), 'stoch_slow': np.float64(5.0)}

Объ

...

Модель tsmixer_full_BTC-USD.pt загружена.
Загружены лучшие веса из tsmixer_full_BTC-USD.pt
ИНФОРМАЦИЯ О МОДЕЛИ
Всего параметров:      305,113
Обучаемых параметров:  305,113
Устройство:            cpu
Веса классов для Loss: 0 (Down): 0.9831, 1 (Up): 1.0174
✓ Optimizer, loss function и evaluation function настроены
НАЧАЛО ОБУЧЕНИЯ
ОБУЧЕНИЕ ЗАВЕРШЕНО. Лучшая точность: 0.0000
ЗАГРУЖАЕМ ЛУЧШИЕ ВЕСА ПЕРЕД ФИНАЛЬНОЙ ОЦЕНКОЙ
Загружены лучшие веса из tsmixer_full_BTC-USD.pt

ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ
Test Accuracy: 0.9623 (96.23%)
Baseline (majority class): 0.5034 (50.34%)
Улучшение над baseline: 45.89%

----------------------------------------------------------------------
CLASSIFICATION REPORT
----------------------------------------------------------------------
              precision    recall  f1-score   support

    Down (0)     0.9658    0.9592    0.9625       147
      Up (1)     0.9589    0.9655    0.9622       145

    accuracy                         0.9623       292
   mac

Backtest.run:   0%|          | 0/291 [00:00<?, ?bar/s]

Start                     2024-05-25 00:00:00
End                       2025-03-12 00:00:00
Duration                    291 days 00:00:00
Exposure Time [%]                   74.657534
Equity Final [$]              10865120.754766
Equity Peak [$]               10887462.894828
Commissions [$]                1141685.721797
Return [%]                         986.512075
Buy & Hold Return [%]               20.870883
Return (Ann.) [%]                 1872.618931
Volatility (Ann.) [%]              671.890174
CAGR [%]                          1892.936756
Sharpe Ratio                         2.787091
Sortino Ratio                      421.254003
Calmar Ratio                       798.810444
Alpha [%]                          975.361555
Beta                                 0.534262
Max. Drawdown [%]                   -2.344259
Avg. Drawdown [%]                   -0.333495
Max. Drawdown Duration        9 days 00:00:00
Avg. Drawdown Duration        4 days 00:00:00
# Trades                          

'risk_adjusted_return = -849995983.95'

Финальный ROC AUC на тесте: 0.9921
